In [ ]:
import os
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from dotenv import load_dotenv
import json
from pathlib import Path
import re
from IPython.display import display
import unicodedata
from difflib import SequenceMatcher

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import time
from datetime import datetime, timezone

import random

In [ ]:
df_games = pd.read_json("igdb_games_raw.json")

# mapping correspondencia IGDB - Steam
df_steam_map = pd.read_json("steam_map.json")

MANIFEST_PATH = Path("steam_etl") / "manifest.csv"
MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
df_steam_manifest = pd.read_csv(MANIFEST_PATH)

"""# mapping correspondencia Steam - ITAD
df_itad_mapping = pd.read_csv("itad_mapping.csv")
df_itad_prices = pd.read_csv("itad_prices.csv")

HISTORY_CSV_PATH = Path("itad_etl") / "itad_price_history.csv"
df_itad_history = pd.read_csv(HISTORY_CSV_PATH)"""

In [ ]:
df_games["first_release_date"] = pd.to_datetime(
    df_games["first_release_date"],
    unit="s",
    errors="coerce"
)

In [ ]:
df_games["first_release_date"].describe()

In [ ]:
df_games["release_year"] =  df_games["first_release_date"].dt.year

In [ ]:
game_type_names = {
    0: "main_game",             # Juego principal
    1: "dlc_addon",             # DLC o contenido adicional
    2: "expansion",             # Expansión
    3: "bundle",                # Paquete de juegos
    4: "standalone_expansion",  # Expansión independiente
    5: "mod",                   # Modificación
    6: "episode",               # Episodio
    7: "season",                # Temporada
    8: "remake",                # Remake
    9: "remaster",              # Remasterización
    10: "expanded_game",        # Juego ampliado
    11: "port",                 # Adaptación a otra plataforma
    12: "fork",                 # Derivado de otro juego
    13: "pack",                 # Pack
    14: "update"                # Actualización
}

df_games["game_type_name"] = df_games["game_type"].map(game_type_names)

df_games["game_type_name"].value_counts(dropna=False)

In [ ]:
# 1. Preparar los mapeos IGDB-Steam
mapping = df_steam_map.copy()

mapping["appid"] = pd.to_numeric(
    mapping["uid"], errors="coerce"
)

mapping = mapping.dropna(subset=["game", "appid"]).copy()
mapping["game"] = mapping["game"].astype(int)
mapping["appid"] = mapping["appid"].astype("int64")

mapping = (
    mapping.drop_duplicates(subset="game")
)

# 2. Unir la información de IGDB con el appid de Steam
candidates = df_games.copy()
candidates["id"] = candidates["id"].astype(int)

candidates = candidates.merge(
    mapping[["game", "appid"]].rename(columns={"game": "id"}),
    on="id",
    how="inner",
    validate="one_to_one"
)

# 3. Preparar variables de estratificación
candidates["release_date"] = pd.to_datetime(
    candidates["first_release_date"],
    unit="s",
    errors="coerce",
    utc=True
)

candidates["release_year"] = candidates["release_date"].dt.year

print("Juegos candidatos:", len(candidates))
print("Años:", candidates["release_year"].min(),
      "-", candidates["release_year"].max())
print("Tipos de juego:", candidates["game_type_name"].nunique())

In [ ]:
candidates_main = candidates.loc[
    candidates["game_type_name"].eq("main_game")
].copy()

candidates_main = (
    candidates_main
    .drop_duplicates(subset="appid")
    .reset_index(drop=True)
)

print("Juegos candidatos:", len(candidates))
print("Main games:", len(candidates_main))
print("AppID únicos:", candidates_main["appid"].nunique())

In [ ]:
candidates_main.isnull().sum()

In [ ]:
Listado de videojuegos y clasificación

Minecraft, PUBG, Black Myth: Wukong, Palworld, Cyberpunk 2077 o Elden Ring

## A. Acción, aventura y mundo abierto

Grand Theft Auto V	Éxito comercial extraordinario

Red Dead Redemption 2	Éxito comercial y crítico

The Witcher 3: Wild Hunt	Gran éxito y valoración

Cyberpunk 2077	Gran popularidad y evolución de reputación

The Elder Scrolls V: Skyrim	Longevidad y éxito histórico

Fallout 4	Éxito comercial

Fallout: New Vegas	Reconocimiento crítico

Hogwarts Legacy	Gran éxito comercial

Elden Ring	Éxito comercial y crítico

Baldur's Gate 3	Éxito comercial y crítico

God of War	Reconocimiento y ventas

Horizon Zero Dawn	Éxito comercial

Ghost of Tsushima	Reconocimiento y ventas

Assassin's Creed Odyssey	Franquicia de gran alcance

Assassin's Creed Valhalla	Éxito comercial

Kingdom Come: Deliverance II	Éxito reciente

Sekiro: Shadows Die Twice	Reconocimiento crítico

Dark Souls III	Relevancia histórica del género

Monster Hunter: World	Éxito comercial

Monster Hunter Rise	Éxito comercial

## B. Shooters y acción multijugador

Counter-Strike 2	Enorme volumen de jugadores y reseñas
Counter-Strike: Global Offensive	Relevancia histórica
PUBG: BATTLEGROUNDS	Récord histórico de jugadores
Helldivers 2	Éxito reciente
DOOM (2016)	Relevancia histórica y crítica
DOOM Eternal	Reconocimiento crítico
Warhammer 40,000: Space Marine 2	Éxito reciente
Call of Duty: Modern Warfare (2019)	Franquicia de gran alcance
Tom Clancy's Rainbow Six Siege	Longevidad y comunidad
Left 4 Dead 2	Éxito histórico en Steam
Apex Legends	Popularidad multijugador
Team Fortress 2	Longevidad y relevancia
Destiny 2	Juego como servicio
Warframe	Longevidad y comunidad
Black Myth: Wukong	Éxito comercial y pico de jugadores
Borderlands 2	Éxito comercial e histórico

##C. Estrategia y gestión

Age of Empires II: Definitive Edition	Clásico y longevidad
Age of Empires IV	Éxito de franquicia
Age of Empires III: Definitive Edition	Representación de la saga
Sid Meier's Civilization V	Gran relevancia histórica
Sid Meier's Civilization VI	Éxito comercial
Sid Meier's Civilization VII	Entrega reciente
Total War: Rome II	Relevancia histórica
Total War: Warhammer II	Éxito de la saga
Total War: Warhammer III	Entrega reciente
Total War: Three Kingdoms	Éxito comercial
Crusader Kings III	Éxito de estrategia
Europa Universalis IV	Longevidad
Stellaris	Longevidad
XCOM 2	Reconocimiento crítico
RimWorld	Éxito Indie y de simulación
Factorio	Éxito Indie y de gestión
Cities: Skylines	Éxito de simulación
Anno 1800	Éxito de estrategia y gestión

## D. RPG, simulación y supervivencia

Stardew Valley	Éxito Indie extraordinario
Terraria	Ventas y popularidad
Minecraft	Éxito comercial histórico
Rust	Gran popularidad en Steam
Valheim	Éxito de supervivencia
ARK: Survival Evolved	Éxito comercial
Project Zomboid	Popularidad y longevidad
Euro Truck Simulator 2	Éxito de simulación
The Sims 4	Franquicia de gran alcance
Palworld	Éxito reciente
Don't Starve Together	Éxito Indie multijugador
Subnautica	Éxito comercial y crítico
The Forest	Éxito de supervivencia
Satisfactory	Éxito Indie de gestión
Monster Hunter Wilds	Éxito de franquicia reciente

## E. Plataformas, puzles, lucha, carreras y otros

Portal	Reconocimiento crítico
Portal 2	Reconocimiento crítico
Hollow Knight	Éxito Indie y crítico
Hades	Éxito Indie y crítico
Hades II	Éxito reciente
Vampire Survivors	Éxito Indie
Celeste	Reconocimiento crítico
Cuphead	Éxito Indie y artístico
Dead Cells	Éxito Indie
Street Fighter 6	Relevancia en lucha
Tekken 7	Relevancia en lucha
Forza Horizon 5	Éxito de carreras
Assetto Corsa	Popularidad en simulación
Need for Speed Heat	Representación de carreras
EA SPORTS FC 25	Representación deportiva
Football Manager 2024	Éxito de simulación deportiva

In [ ]:
iconic_games = [
    # Acción, aventura y mundo abierto
    "Grand Theft Auto V",
    "Red Dead Redemption 2",
    "The Witcher 3: Wild Hunt",
    "Cyberpunk 2077",
    "The Elder Scrolls V: Skyrim",
    "Fallout 4",
    "Fallout: New Vegas",
    "Hogwarts Legacy",
    "Elden Ring",
    "Baldur's Gate III",
    "God of War",
    "Horizon Zero Dawn",
    "Ghost of Tsushima",
    "Assassin's Creed Odyssey",
    "Assassin's Creed Valhalla",
    "Sekiro: Shadows Die Twice",
    "Dark Souls III",
    "Monster Hunter: World",
    "Monster Hunter Rise",

    # Shooters y acción
    "Counter-Strike 2",
    "Counter-Strike: Global Offensive",
    "PUBG: BATTLEGROUNDS",
    "HELLDIVERS 2",
    "DOOM",
    "DOOM Eternal",
    "Warhammer 40,000: Space Marine",
    "Warhammer 40,000: Space Marine II",
    "Call of Duty: Modern Warfare",
    "Rainbow Six Siege",
    "Left 4 Dead 2",
    "Apex Legends",
    "Team Fortress 2",
    "Destiny 2",
    "Warframe",
    "Black Myth: Wukong",
    "Borderlands 2",

    # Estrategia
    "Age of Empires II: Definitive Edition",
    "Age of Empires IV",
    "Age of Empires III: Definitive Edition",
    "Sid Meier's Civilization V",
    "Sid Meier's Civilization VI",
    "Sid Meier's Civilization VII",
    "Total War: Rome II",
    "Total War: Warhammer II",
    "Total War: Warhammer III",
    "Total War: Three Kingdoms",
    "Crusader Kings III",
    "Europa Universalis IV",
    "Stellaris",
    "XCOM 2",
    "RimWorld",
    "Factorio",
    "Cities: Skylines",
    "Anno 1800",

    # Simulación y supervivencia
    "Stardew Valley",
    "Terraria",
    "Minecraft",
    "Rust",
    "Valheim",
    "ARK: Survival Evolved",
    "Project Zomboid",
    "Euro Truck Simulator 2",
    "The Sims 4",
    "Palworld",
    "Don't Starve Together",
    "Subnautica",
    "The Forest",
    "Satisfactory",
    "Monster Hunter Wilds",

    # Otros géneros
    "Portal",
    "Portal 2",
    "Hollow Knight",
    "Hollow Knight: Silksong",
    "Hades",
    "Hades II",
    "Vampire Survivors",
    "Celeste",
    "Cuphead",
    "Dead Cells",
    "Street Fighter 6",
    "Tekken 7",
    "Forza Horizon 5",
    "Assetto Corsa",
    "Need for Speed: Heat",
    "EA SPORTS FC 25",
    "Football Manager 2024",

    # juegos adicionales
    "Elden Ring Nightreign",
    "God of War Ragnarök",
    "Call of Duty: Modern Warfare II",
    "Call of Duty: Modern Warfare III",
    "Sid Meier's Civilization VI",
    "Sid Meier's Civilization VII",
    "Total War: Warhammer III",
    "Cities: Skylines II",
    "Subnautica 2",
    "Assetto Corsa Competizione",
    "Assetto Corsa EVO"
]

In [ ]:
# buscamos coincidencias en candidates_main

matches = []

for game in iconic_games:

    mask = candidates_main["name"].str.contains(
        re.escape(game),
        case=False,
        na=False,
        regex=True
    )

    found = candidates_main.loc[mask].copy()

    if not found.empty:
        found["search_term"] = game
        matches.append(found)

if matches:
    df_iconic_matches = pd.concat(
        matches,
        ignore_index=True
    )

    display(
        df_iconic_matches[
            [
                col for col in [
                    "search_term",
                    "id",
                    "name",
                    "first_release_date",
                    "game_type"
                ]
                if col in df_iconic_matches.columns
            ]
        ].sort_values("search_term")
    )
else:
    df_iconic_matches = pd.DataFrame()
    print("No se han encontrado coincidencias.")

In [ ]:
df_iconic_matches[["search_term", "id", "name"]]

In [ ]:
unique_search_term = df_iconic_matches["search_term"].unique()

for search_term in unique_search_term:
    if len(df_iconic_matches[["search_term", "id", "name"]][df_iconic_matches["search_term"] == search_term]) > 1:
        display(df_iconic_matches[["search_term", "id", "name"]][df_iconic_matches["search_term"] == search_term])

In [ ]:
# Normalizamos los nombres
candidates_main["name_normalized"] = (
    candidates_main["name"]
    .str.strip()
    .str.casefold()
)

titles_normalized = {
    title.casefold(): title
    for title in iconic_games
}

# Coincidencias exactas
df_iconic_exact = candidates_main[
    candidates_main["name_normalized"].isin(titles_normalized)
].copy()

df_iconic_exact["matched_title"] = (
    df_iconic_exact["name_normalized"].map(titles_normalized)
)

df_iconic_exact = (
    df_iconic_exact
    .drop_duplicates(subset="id")
    .sort_values("name")
    .reset_index(drop=True)
)

display(
    df_iconic_exact[
        ["id", "name", "matched_title", "first_release_date"]
    ]
)

print("Títulos buscados:", len(iconic_games))
print("Juegos encontrados:", len(df_iconic_exact))

In [ ]:
found_titles = set(df_iconic_exact["matched_title"].str.casefold())

missing_titles = [
    title for title in iconic_games
    if title.casefold() not in found_titles
]

variant_matches = []

for title in missing_titles:
    # Coincidencia por nombre completo como expresión independiente
    pattern = (
        r"(?<!\w)"
        + re.escape(title)
        + r"(?!\w)"
    )

    found = candidates_main[
        candidates_main["name"].str.contains(
            pattern,
            case=False,
            na=False,
            regex=True
        )
    ].copy()

    if not found.empty:
        found["matched_title"] = title
        variant_matches.append(found)

if variant_matches:
    df_iconic_variants = pd.concat(
        variant_matches,
        ignore_index=True
    ).drop_duplicates(subset=["id", "matched_title"])

    display(
        df_iconic_variants[
            ["id", "name", "matched_title", "first_release_date"]
        ].sort_values(["matched_title", "name"])
    )
else:
    df_iconic_variants = pd.DataFrame()
    print("No se han encontrado variantes adicionales.")

In [ ]:
found_titles = set(
    df_iconic_exact["matched_title"].str.casefold()
)

missing_titles = [
    title for title in iconic_games
    if title.casefold() not in found_titles
]

print("Títulos sin coincidencia exacta:", len(missing_titles))

for title in missing_titles:
    print("-", title)

In [ ]:
display(
    df_iconic_variants[
        ["matched_title", "name", "id", "first_release_date"]
    ]
    .sort_values(["matched_title", "name"])
    .reset_index(drop=True)
)

podemos ampliar la lista

In [ ]:
# Auditamos candidates_main
# Porcentaje de nulos en la población main_game

sampling_columns = [
    "genres",
    "platforms",
    "game_modes",
    "player_perspectives",
    "first_release_date",
    "rating",
    "rating_count",
    "popularity",
    "hypes",
    "follows"
]

sampling_columns = [
    col for col in sampling_columns
    if col in candidates_main.columns
]

df_sampling_audit = pd.DataFrame({
    "nulls": candidates_main[sampling_columns].isna().sum(),
    "null_pct": (
        candidates_main[sampling_columns].isna().mean() * 100
    ),
    "non_null": candidates_main[sampling_columns].notna().sum()
})

display(
    df_sampling_audit.sort_values(
        "null_pct",
        ascending=False
    )
)

In [ ]:
print("Total main games:", len(candidates_main))

print(
    "Juegos sin género:",
    candidates_main["genres"].isna().sum()
)

print(
    "Juegos sin fecha:",
    candidates_main["first_release_date"].isna().sum()
)

print(
    "Juegos sin plataforma:",
    candidates_main["platforms"].isna().sum()
)

In [ ]:
# Número de coincidencias y juegos únicos
print("Filas en df_iconic_matches:", len(df_iconic_matches))
print("Juegos únicos por IGDB ID:", df_iconic_matches["id"].nunique())
print("IDs duplicados por búsquedas solapadas:",
      df_iconic_matches.duplicated(subset="id").sum())

# Resultados por término de búsqueda
df_iconic_summary = (
    df_iconic_matches
    .groupby("search_term")
    .agg(
        matches=("id", "size"),
        unique_games=("id", "nunique")
    )
    .sort_values("matches", ascending=False)
)

display(df_iconic_summary)

In [ ]:
df_iconic_review = (
    df_iconic_matches
    .groupby(["id", "name"], as_index=False)
    .agg(
        search_terms=(
            "search_term",
            lambda x: " | ".join(sorted(set(x)))
        )
    )
)

# Añadimos la fecha de lanzamiento
df_iconic_review = df_iconic_review.merge(
    candidates_main[["id", "first_release_date"]],
    on="id",
    how="left"
)

display(
    df_iconic_review
    .sort_values("name")
    .reset_index(drop=True)
)

In [ ]:
df_iconic_steam = df_iconic_exact.merge(
    df_steam_map[["game", "uid"]],
    left_on="id",
    right_on="game",
    how="left"
)

df_iconic_steam_summary = (
    df_iconic_steam
    .groupby(["id", "name"], as_index=False)
    .agg(
        steam_appids=("uid", "nunique"),
        steam_appid=("uid", "first")
    )
)

print("Juegos icónicos:", len(df_iconic_steam_summary))
print(
    "Con correspondencia Steam:",
    (df_iconic_steam_summary["steam_appids"] > 0).sum()
)
print(
    "Sin correspondencia Steam:",
    (df_iconic_steam_summary["steam_appids"] == 0).sum()
)

display(
    df_iconic_steam_summary
    .sort_values(["steam_appids", "name"], ascending=[True, True])
)

In [ ]:
variant_searches = {
    "Horizon Zero Dawn": "Horizon Zero Dawn",
    "Ghost of Tsushima": "Ghost of Tsushima",
    "Age of Empires II: Definitive Edition": "Age of Empires II",
    "Age of Empires III: Definitive Edition": "Age of Empires III",
    "Total War: Rome II": "Total War: Rome II",
    "Don't Starve Together": "Don't Starve",
    "Tekken 7": "Tekken 7",
    "Need for Speed Heat": "Need for Speed"
}

variant_results = []

for expected_title, fragment in variant_searches.items():

    found = candidates_main[
        candidates_main["name"].str.contains(
            re.escape(fragment),
            case=False,
            na=False,
            regex=True
        )
    ].copy()

    if not found.empty:
        found["expected_title"] = expected_title
        variant_results.append(found)

if variant_results:
    df_iconic_candidates = (
        pd.concat(variant_results, ignore_index=True)
        .drop_duplicates(subset="id")
        .sort_values(["expected_title", "name"])
        .reset_index(drop=True)
    )

    display(
        df_iconic_candidates[
            ["expected_title", "id", "name", "first_release_date"]
        ]
    )
else:
    df_iconic_candidates = pd.DataFrame()
    print("No se han encontrado variantes.")

In [ ]:
pending_iconic = [
    "Horizon",
    "Ghost of Tsushima",
    "Age of Empires II",
    "Age of Empires III",
    "Don't Starve Together",
    "Tekken"
]

pending_results = []

for term in pending_iconic:
    found = candidates_main[
        candidates_main["name"].str.contains(
            re.escape(term),
            case=False,
            na=False,
            regex=True
        )
    ].copy()

    if not found.empty:
        found["search_term"] = term
        pending_results.append(found)

if pending_results:
    df_iconic_pending = (
        pd.concat(pending_results, ignore_index=True)
        .drop_duplicates(subset="id")
        .sort_values(["search_term", "name"])
        .reset_index(drop=True)
    )

    display(
        df_iconic_pending[
            ["search_term", "id", "name", "first_release_date"]
        ]
    )
else:
    df_iconic_pending = pd.DataFrame()
    print("No se han encontrado coincidencias.")

In [ ]:
# Normalizar nombres
candidates_main["name_normalized"] = (
    candidates_main["name"]
    .str.strip()
    .str.casefold()
)

# Diccionario de títulos normalizados
titles_normalized = {
    title.casefold(): title
    for title in iconic_games
}

# Buscar coincidencias exactas
df_iconic_exact = candidates_main[
    candidates_main["name_normalized"].isin(titles_normalized)
].copy()

df_iconic_exact["matched_title"] = (
    df_iconic_exact["name_normalized"].map(titles_normalized)
)

df_iconic_exact = (
    df_iconic_exact
    .drop_duplicates(subset="id")
    .sort_values("name")
    .reset_index(drop=True)
)

print("Títulos en iconic_games:", len(iconic_games))
print("Juegos encontrados:", len(df_iconic_exact))
print("IDs IGDB únicos:", df_iconic_exact["id"].nunique())

display(
    df_iconic_exact[
        ["id", "name", "matched_title", "first_release_date"]
    ]
)

In [ ]:
# Relacionar los juegos icónicos con sus AppIDs de Steam
df_iconic_steam = df_iconic_exact.merge(
    df_steam_map[["game", "uid"]],
    left_on="id",
    right_on="game",
    how="left"
)

# Resumen por juego
df_iconic_steam_summary = (
    df_iconic_steam
    .groupby(["id", "name"], as_index=False)
    .agg(
        steam_appids=("uid", "nunique"),
        appids=("uid", lambda x: sorted(x.dropna().unique().tolist()))
    )
)

# Clasificación de cobertura
df_iconic_steam_summary["steam_status"] = (
    df_iconic_steam_summary["steam_appids"]
    .map({
        0: "No Steam match",
        1: "One Steam match"
    })
    .fillna("Multiple Steam matches")
)

display(
    df_iconic_steam_summary
    .sort_values(["steam_status", "name"])
)

print("\nResumen de cobertura:")
print(df_iconic_steam_summary["steam_status"].value_counts())

In [ ]:
# Juegos icónicos con exactamente 2 AppIDs
df_iconic_duplicates = (
    df_iconic_steam_summary[
        df_iconic_steam_summary["steam_appids"] == 2
    ]
    .copy()
    .sort_values("name")
    .reset_index(drop=True)
)

print(f"Juegos con 2 AppIDs: {len(df_iconic_duplicates)}")

In [ ]:
for _, game in df_iconic_duplicates.iterrows():

    igdb_id = game["id"]
    game_name = game["name"]

    print("\n" + "=" * 80)
    print(f"JUEGO: {game_name}")
    print(f"IGDB ID: {igdb_id}")
    print("=" * 80)

    # Recuperar sus dos correspondencias
    candidates = (
        df_steam_map[
            df_steam_map["game"] == igdb_id
        ]
        .copy()
        .sort_values("uid")
    )

    # Mostrar toda la información disponible
    display(candidates)

In [ ]:
# Diccionario de selección manual
# Formato: IGDB_ID: UID(APPID)_QUE_CONSERVAMOS

selected_appids = {
    191692: 1364780,    # Street Fighter 6
    71: 400,            # Portal
    16: 22380,          # Fallout: New Vegas
    9630: 377160,       # Fallout 4
    1904: 236850,       # Europa Universalis IV
    7351: 379720,       # Doom
    26855: 588650,      # Dead Cells
    260780: 2519060,    # Call of Duty: Modern Warfare III
    200165: 1962660,    # Call of Duty: Modern Warfare II
    1011: 379880,       # Borderlands 2
    89444: 1657450,     # Assetto Corsa Competizione
    103054: 812140,     # Assassin's Creed Odyssey
    55029: 1466860      # Age of Empires IV
}

In [ ]:
df_steam_map["game"] = pd.to_numeric(
    df_steam_map["game"],
    errors="coerce"
).astype("Int64")

df_steam_map["uid"] = pd.to_numeric(
    df_steam_map["uid"],
    errors="coerce"
).astype("Int64")

selected_appids = {
    int(igdb_id): int(appid)
    for igdb_id, appid in selected_appids.items()
}

In [ ]:
# Comprobar que hemos seleccionado exactamente
# un AppID válido por cada juego duplicado

valid_selections = []

for _, game in df_iconic_duplicates.iterrows():

    igdb_id = game["id"]

    candidates = set(
        df_steam_map.loc[
            df_steam_map["game"] == igdb_id,
            "uid"
        ]
    )

    selected = selected_appids.get(igdb_id)

    if selected is None:
        print(f"FALTA SELECCIÓN: {game['name']}")

    elif selected not in candidates:
        print(
            f"ERROR: {game['name']} - "
            f"AppID {selected} no pertenece a sus candidatos"
        )

    else:
        valid_selections.append({
            "igdb_id": igdb_id,
            "name": game["name"],
            "selected_appid": selected
        })

df_iconic_selections = pd.DataFrame(valid_selections)

print(f"\nSelecciones válidas: {len(df_iconic_selections)}")
print(f"Juegos pendientes: {len(df_iconic_duplicates) - len(valid_selections)}")

In [ ]:
# Separar las correspondencias de juegos duplicados
duplicate_ids = df_iconic_duplicates["id"].tolist()

# Crear mapping de los juegos duplicados
df_duplicate_mapping = df_steam_map[
    df_steam_map["game"].isin(duplicate_ids)
].copy()

# Conservar únicamente el AppID seleccionado
selected_mapping = df_iconic_selections.set_index(
    "igdb_id"
)["selected_appid"].to_dict()

df_duplicate_mapping = df_duplicate_mapping[
    df_duplicate_mapping.apply(
        lambda row: selected_mapping.get(row["game"]) == row["uid"],
        axis=1
    )
]

# Recuperar los juegos que no tenían duplicados
df_single_mapping = df_steam_map[
    ~df_steam_map["game"].isin(duplicate_ids)
].copy()

# Unir ambos mappings
df_steam_map_clean = pd.concat(
    [df_single_mapping, df_duplicate_mapping],
    ignore_index=True
)

# Comprobaciones
print("Mapping original:", len(df_steam_map))
print("Mapping limpio:", len(df_steam_map_clean))

print(
    "Juegos icónicos que siguen teniendo múltiples AppIDs:",
    df_steam_map_clean[
        df_steam_map_clean["game"].isin(duplicate_ids)
    ]["game"].value_counts().gt(1).sum()
)

In [ ]:
# Comprobar duplicados entre los juegos icónicos
df_iconic_final = df_steam_map_clean[
    df_steam_map_clean["game"].isin(
        df_iconic_steam_summary["id"]
    )
]

# AppIDs por juego
final_counts = (
    df_iconic_final
    .groupby("game")["uid"]
    .nunique()
)

print("Distribución final de correspondencias:")
print(final_counts.value_counts().sort_index())

print("\nJuegos con más de un AppID:")
print(final_counts[final_counts > 1])

In [ ]:
df_iconic_final

In [ ]:
df_genres = pd.read_csv("igdb_genres_names.csv")
df_platforms = pd.read_csv("igdb_platforms_names.csv")
df_game_modes = pd.read_csv("igdb_game_modes_names.csv")
df_player_perspectives = pd.read_csv("igdb_player_perspectives_names.csv")

In [ ]:
# Diccionario ID de género -> nombre
genre_lookup = dict(
    zip(df_genres["id"], df_genres["name"])
)

def get_genre_names(value):

    if not isinstance(value, (list, tuple, np.ndarray)):
        return []

    names = []

    for genre in value:

        if isinstance(genre, dict):
            name = genre.get("name")
        elif isinstance(genre, (int, np.integer)):
            name = genre_lookup.get(genre)
        elif isinstance(genre, str) and genre.isdigit():
            name = genre_lookup.get(int(genre))
        else:
            name = genre

        if name is not None:
            names.append(str(name))

    return names


# Trabajar sobre una copia
df_games_clean = df_games.copy()

# Convertir los géneros a nombres
df_games_clean["genres_names"] = (
    df_games_clean["genres"].apply(get_genre_names)
)

# representa aquí la clasificación Indie de IGDB,
# no necesariamente que el desarrollador sea independiente en términos empresariales

# Crear variable indie
df_games_clean["is_indie"] = (
    df_games_clean["genres_names"]
    .apply(lambda x: any(g.strip().lower() == "indie" for g in x))
)

# Eliminar Indie de los géneros
df_games_clean["genres_clean"] = (
    df_games_clean["genres_names"]
    .apply(
        lambda x: [
            g for g in x
            if g.strip().lower() != "indie"
        ]
    )
)

# Comprobaciones
print("Juegos indie:", df_games_clean["is_indie"].sum())
print("Juegos no indie:", (~df_games_clean["is_indie"]).sum())

print("\nDistribución:")
print(df_games_clean["is_indie"].value_counts(normalize=True) * 100)

print("\nEjemplos:")
display(
    df_games_clean[
        ["name", "genres_names", "genres_clean", "is_indie"]
    ].head(10)
)

In [ ]:
# Distribución de años
print("DISTRIBUCIÓN TEMPORAL")
print(
    df_games_clean["release_year"]
    .describe()
)

# Distribución Indie
print("\nDISTRIBUCIÓN INDIE")
print(
    df_games_clean["is_indie"].value_counts()
)

# Distribución de tipos
print("\nDISTRIBUCIÓN GAME TYPE")
print(
    df_games_clean["game_type"].value_counts(dropna=False)
)

# Distribución de géneros
print("\nDISTRIBUCIÓN DE GÉNEROS")

genre_counts = (
    df_games_clean["genres_clean"]
    .explode()
    .value_counts()
)

print(genre_counts)

In [ ]:
# Filtrar únicamente juegos principales
df_main_games = df_games_clean[
    df_games_clean["game_type"] == 0
].copy()

df_main_games.reset_index(drop=True, inplace=True)

print("Juegos Main Game:", len(df_main_games))

print("\nDistribución de Indie:")
print(df_main_games["is_indie"].value_counts())

print("\nPorcentaje:")
print(
    df_main_games["is_indie"]
    .value_counts(normalize=True)
    .mul(100)
)

In [ ]:
CURRENT_YEAR = 2026

print("Años nulos:", df_main_games["release_year"].isna().sum())

print(
    "Juegos con lanzamiento futuro:",
    (df_main_games["release_year"] > CURRENT_YEAR).sum()
)

print(
    "Juegos anteriores a 2000:",
    (df_main_games["release_year"] < 2000).sum()
)

print(
    "Distribución de años:",
    df_main_games["release_year"].value_counts().sort_index()
)

In [ ]:
# IDs de juegos con correspondencia en Steam
steam_game_ids = df_steam_map_clean["game"].unique()

# Añadir indicador de disponibilidad en Steam
df_main_games["has_steam"] = (
    df_main_games["id"].isin(steam_game_ids)
)

print("Juegos Main Game:", len(df_main_games))

print("\nDisponibilidad Steam:")
print(df_main_games["has_steam"].value_counts())

print("\nPorcentaje:")
print(
    df_main_games["has_steam"]
    .value_counts(normalize=True)
    .mul(100)
)

In [ ]:
df_sampling_pool = df_main_games[
    df_main_games["has_steam"]
].copy()

print("Muestreo inicial:", len(df_sampling_pool))

In [ ]:
bins = [
    0,
    2000,
    2010,
    2015,
    2020,
    2026
]

labels = [
    "Pre-2000",
    "2001-2010",
    "2011-2015",
    "2016-2020",
    "2021-2026"
]

df_sampling_pool["release_period"] = pd.cut(
    df_sampling_pool["release_year"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

In [ ]:
print("=== UNIVERSO MAIN GAME + STEAM ===")
print(len(df_sampling_pool))

print("\n=== INDIE ===")
print(df_sampling_pool["is_indie"].value_counts())

print("\n=== PERIODOS ===")
print(df_sampling_pool["release_period"].value_counts(dropna=False))

print("\n=== GÉNEROS ===")
print(
    df_sampling_pool["genres_clean"]
    .explode()
    .value_counts()
)

print("\n=== FECHAS ===")
print(df_sampling_pool["release_year"].describe())

print("\n=== GÉNEROS VACÍOS ===")
print(
    df_sampling_pool["genres_clean"]
    .apply(lambda x: len(x) == 0)
    .sum()
)

In [ ]:
# Diagnóstico de fechas

print("=== RELEASE YEAR ===")

print(
    "Fechas nulas:",
    df_sampling_pool["release_year"].isna().sum()
)

print(
    "Fechas anteriores a 1947:",
    (df_sampling_pool["release_year"] < 1947).sum()
)

print(
    "Fechas posteriores a 2026:",
    (df_sampling_pool["release_year"] > 2026).sum()
)

print("\nDistribución de años futuros:")

print(
    df_sampling_pool.loc[
        df_sampling_pool["release_year"] > 2026,
        "release_year"
    ].value_counts().sort_index()
)

In [ ]:
# Diagnóstico de géneros

print("=== GENRES ===")

print(
    "Juegos sin género:",
    df_sampling_pool["genres_clean"]
    .apply(lambda x: not isinstance(x, list) or len(x) == 0)
    .sum()
)

print("\nDistribución completa de géneros:")

genre_counts = (
    df_sampling_pool["genres_clean"]
    .explode()
    .value_counts()
)

display(genre_counts.to_frame("games"))

# Limpieza

In [ ]:
# Trabajamos sobre una copia del universo actual
df_audit = df_sampling_pool.copy()

# Patrones sospechosos
name_patterns = [
    r"\bdemo\b",
    r"\bbeta\b",
    r"\bopen beta\b",
    r"\bclosed beta\b",
    r"\bplaytest\b",
    r"\balpha\b",
    r"\btechnical stress test\b",
    r"\bstress test\b",
    r"\btest server\b",
    r"\bdedicated server\b",
    r"\bsoundtrack\b",
    r"\boriginal soundtrack\b",
    r"\bost\b",
    r"\bdlc\b",
    r"\bexpansion\b",
    r"\bexpansion pack\b",
    r"\btrial\b"
]

name_regex = "|".join(name_patterns)

# Detectar coincidencias
df_audit["name_suspicious"] = (
    df_audit["name"]
    .fillna("")
    .str.contains(name_regex, case=False, regex=True)
)

# Recuperar candidatos
df_suspicious_names = (
    df_audit[df_audit["name_suspicious"]]
    [["id", "name", "release_year", "genres_clean", "is_indie"]]
    .sort_values("name")
    .copy()
)

print("Juegos sospechosos:", len(df_suspicious_names))

display(df_suspicious_names)

391923	Horror Engine: Tech Demo
303557	Hybrid Miniature Golf Beta
83885	Just Survive Test Server
133564	MetaWare High School Demo
93808	Natural Selection 2 - Deluxe DLC
76697	Nodge Pre-Alpha ALPHA
265442	PsionicAwake Playtest
93812	Railworks Class 20 Pack DLC
93845	Railworks Class56 DLC
200168	Stress Test
51433	Strike Vector EX Open Beta
405622	Sudoku: Unlimited Expansion
111208	The Anomaly Demo
86625	Ancestors Legacy Multiplayer Open Beta
266990	AploVVare Collection:18+ DLC	[NO RELEASE_YEAR]
100573	Blood Ancestors: Open Alpha
36473	Hell Warders - Beta

In [ ]:
# IDs de los 17 juegos que hemos decidido excluir
ids_to_remove = [
    391923, 303557, 83885, 133564, 93808,
    76697, 265442, 93812, 93845, 200168,
    51433, 405622, 111208, 86625, 266990,
    100573, 36473
]

# Comprobaciones previas
print("IDs indicados:", len(ids_to_remove))
print("IDs únicos:", len(set(ids_to_remove)))

# Guardamos los registros que vamos a excluir
df_excluded_suspicious_names = df_sampling_pool[
    df_sampling_pool["id"].isin(ids_to_remove)
].copy()

df_excluded_suspicious_names["exclusion_reason"] = (
    "Alpha, beta, demo, DLC, expansion, playtest o test server"
)

# Creamos el nuevo pool sin modificar el original
df_sampling_pool_clean = df_sampling_pool[
    ~df_sampling_pool["id"].isin(ids_to_remove)
].copy()

# Comprobamos si todos los IDs estaban realmente en el sampling pool
ids_found = set(df_excluded_suspicious_names["id"])
ids_not_found = set(ids_to_remove) - ids_found

print(f"\nJuegos originales: {len(df_sampling_pool):,}")
print(f"Juegos excluidos: {len(df_excluded_suspicious_names):,}")
print(f"Juegos restantes: {len(df_sampling_pool_clean):,}")
print(f"IDs no encontrados en el pool: {len(ids_not_found)}")

if ids_not_found:
    print("IDs no encontrados:", sorted(ids_not_found))

# Comprobamos que no queda ninguno de los IDs excluidos
remaining_suspicious = df_sampling_pool_clean[
    df_sampling_pool_clean["id"].isin(ids_to_remove)
]

print("\nJuegos excluidos que todavía aparecen:", len(remaining_suspicious))

In [ ]:
# Revisión de lo que hemos excluído
df_excluded_suspicious_names[
    ["id", "name", "release_year", "game_type", "exclusion_reason"]
].sort_values("name")

In [ ]:
# Guardamos el resultado de la limpieza
"""df_excluded_suspicious_names.to_csv(
    "excluded_suspicious_names.csv",
    index=False
)"""

In [ ]:
# Diagnóstico de fechas
date_audit = {
    "total_games": len(df_audit),
    "null_release_year": df_audit["release_year"].isna().sum(),
    "future_release_year": (
        df_audit["release_year"] > CURRENT_YEAR
    ).sum(),
    "before_1947": (
        df_audit["release_year"] < 1947
    ).sum(),
    "valid_release_year": (
        df_audit["release_year"].between(1947, CURRENT_YEAR)
    ).sum()
}

display(
    pd.Series(date_audit, name="count").to_frame()
)

In [ ]:
df_future_games = (
    df_audit[
        df_audit["release_year"] > CURRENT_YEAR
    ][["id", "name", "release_year", "genres_clean", "is_indie"]]
    .sort_values("release_year")
)

display(df_future_games)

Sin fecha: excluir de la muestra principal.

Fecha posterior a 2026: excluir.

Fecha anterior a 1947: revisar individualmente.

Fecha válida entre 1947 y 2026: conservar.

In [ ]:
df_past_games = (
    df_audit[
        df_audit["release_year"] < 1948
    ][["id", "name", "release_year", "genres_clean", "is_indie"]]
    .sort_values("release_year")
)

display(df_past_games)

In [ ]:
# Aseguramos que release_year tenga formato numérico
df_sampling_pool_clean["release_year"] = pd.to_numeric(
    df_sampling_pool_clean["release_year"],
    errors="coerce"
)

# Identificamos los juegos que vamos a excluir
mask_no_release_year = df_sampling_pool_clean["release_year"].isna()
mask_future_release = df_sampling_pool_clean["release_year"] >= 2027

# Guardamos los excluidos
df_excluded_dates = df_sampling_pool_clean[
    mask_no_release_year | mask_future_release
].copy()

# Añadimos el motivo de exclusión
df_excluded_dates["exclusion_reason"] = "Future release year"

df_excluded_dates.loc[
    mask_no_release_year.loc[df_excluded_dates.index],
    "exclusion_reason"
] = "Missing release year"

# Creamos el nuevo universo
df_sampling_pool_clean = df_sampling_pool_clean[
    ~(mask_no_release_year | mask_future_release)
].copy()

# Resumen
print(f"Juegos excluidos por fecha: {len(df_excluded_dates):,}")
print(f"  - Sin fecha: {mask_no_release_year.sum():,}")
print(f"  - Año 2027 o posterior: {mask_future_release.sum():,}")
print(f"\nUniverso restante: {len(df_sampling_pool_clean):,}")

# Validaciones
print("\nNulos en release_year:",
      df_sampling_pool_clean["release_year"].isna().sum())

print("Años >= 2027:",
      (df_sampling_pool_clean["release_year"] >= 2027).sum())

print("Año mínimo:",
      df_sampling_pool_clean["release_year"].min())

print("Año máximo:",
      df_sampling_pool_clean["release_year"].max())

In [ ]:
# Juegos sin géneros identificados
df_no_genres = df_audit[
    df_audit["genres_clean"].apply(
        lambda x: not isinstance(x, (list, tuple)) or len(x) == 0
    )
].copy()

print("Juegos sin género:", len(df_no_genres))

# Distribución de géneros
genre_distribution = (
    df_audit["genres_clean"]
    .explode()
    .dropna()
    .value_counts()
)

display(genre_distribution.to_frame("games"))

No queremos eliminar de la muestra los juegos sin género ya que perderíamos un 17% de la muestra. En su lugar, les asignaremos la etiqueta Unknown

In [ ]:
# Conservamos la columna original
df_sampling_pool_clean["genres_for_analysis"] = (
    df_sampling_pool_clean["genres_clean"].copy()
)

# Sustituimos listas vacías o valores nulos por Unknown
def assign_unknown_genre(genres):
    if not isinstance(genres, (list, tuple, set)):
        return ["Unknown"]

    if len(genres) == 0:
        return ["Unknown"]

    return list(genres)

df_sampling_pool_clean["genres_for_analysis"] = (
    df_sampling_pool_clean["genres_for_analysis"]
    .apply(assign_unknown_genre)
)

# Comprobaciones
games_without_genre = df_sampling_pool_clean[
    df_sampling_pool_clean["genres_for_analysis"].apply(
        lambda x: x == ["Unknown"]
    )
]

print("Juegos clasificados como Unknown:",
      len(games_without_genre))

print("\nDistribución de géneros:")
genre_distribution = (
    df_sampling_pool_clean["genres_for_analysis"]
    .explode()
    .value_counts()
)

display(genre_distribution)

In [ ]:
# Correspondencias múltiples: un juego IGDB -> varios AppIDs
mapping_by_game = (
    df_steam_map_clean
    .groupby("game")
    .agg(
        appid_count=("uid", "nunique"),
        appids=("uid", list),
        steam_names=("name", list)
    )
    .reset_index()
)

df_multiple_appids = mapping_by_game[
    mapping_by_game["appid_count"] > 1
].copy()

print("Juegos con múltiples AppIDs:", len(df_multiple_appids))

display(df_multiple_appids)

In [ ]:
# Un mismo AppID asociado a varios juegos IGDB
mapping_by_appid = (
    df_steam_map_clean
    .groupby("uid")
    .agg(
        igdb_count=("game", "nunique"),
        igdb_ids=("game", list),
        steam_names=("name", list)
    )
    .reset_index()
)

df_shared_appids = mapping_by_appid[
    mapping_by_appid["igdb_count"] > 1
].copy()

print("AppIDs asociados a varios juegos IGDB:", len(df_shared_appids))

display(df_shared_appids)

In [ ]:
# Comprobar los juegos icónicos
iconic_ids = set(df_iconic_final["id"])

pool_ids = set(df_sampling_pool["id"])

iconic_in_pool = iconic_ids.intersection(pool_ids)
iconic_missing = iconic_ids - pool_ids

print("Juegos icónicos:", len(iconic_ids))
print("Presentes en sampling pool:", len(iconic_in_pool))
print("Ausentes:", len(iconic_missing))

In [ ]:
# IDs IGDB de los juegos icónicos
iconic_ids = set(df_iconic_final["id"].dropna().astype(int))

# IDs presentes en cada universo
main_game_ids = set(df_main_games["id"].dropna().astype(int))
sampling_pool_ids = set(df_sampling_pool["id"].dropna().astype(int))

# Separar los icónicos según su presencia
iconic_in_pool = iconic_ids & sampling_pool_ids
iconic_not_in_pool = iconic_ids - sampling_pool_ids
iconic_in_main_games = iconic_ids & main_game_ids
iconic_not_main_games = iconic_ids - main_game_ids

print("Total de IDs icónicos:", len(iconic_ids))
print("Presentes en Main Games:", len(iconic_in_main_games))
print("Presentes en sampling pool:", len(iconic_in_pool))
print("Ausentes de Main Games:", len(iconic_not_main_games))
print("Ausentes del sampling pool:", len(iconic_not_in_pool))

In [ ]:
df_iconic_missing = df_iconic_final[
    df_iconic_final["id"].isin(iconic_not_in_pool)
].copy()

# Comprobar si tienen correspondencia en el mapping de Steam
mapped_igdb_ids = set(
    df_steam_map_clean["game"].dropna().astype(int)
)

df_iconic_missing["is_main_game"] = (
    df_iconic_missing["id"].isin(main_game_ids)
)

df_iconic_missing["has_steam_mapping"] = (
    df_iconic_missing["id"].isin(mapped_igdb_ids)
)

df_iconic_missing[
    ["id", "game", "name", "uid", "is_main_game", "has_steam_mapping"]
].sort_values("id")

In [ ]:
df_iconic_missing[
    ["id", "game", "name", "uid", "is_main_game", "has_steam_mapping"]
][df_iconic_missing["is_main_game"] == True]

In [ ]:
df_iconic_missing[
    ["id", "game", "name", "uid", "is_main_game", "has_steam_mapping"]
][df_iconic_missing["is_main_game"] == False]

### Tratamiento de los videojuegos icónicos

Durante la construcción del universo de análisis se identificaron 86 videojuegos considerados icónicos por su relevancia histórica, comercial y/o cultural dentro de la industria. Esta selección complementaria incluye títulos que han tenido un impacto significativo en la evolución del mercado, han alcanzado una gran difusión entre los jugadores, han contribuido a definir géneros o tendencias, o se han consolidado como referentes dentro de la industria.

Entre los títulos identificados se encuentran, por ejemplo, *The Witcher 3: Wild Hunt*, *Rust*, *Fallout: New Vegas*, *Portal*, *DOOM*, *Stardew Valley*, *Terraria*, *Hollow Knight*, *Red Dead Redemption 2*, *Elden Ring*, *Baldur's Gate 3* y *Cyberpunk 2077*.

Durante la comprobación de su presencia en el universo de muestreo se observó que una parte importante de estos títulos no estaba incluida en el conjunto de juegos clasificados como `Main Game` por IGDB o no aparecía en el mapping utilizado para construir el universo de análisis. Esta situación pone de manifiesto que las clasificaciones y relaciones de IGDB no siempre representan de forma intuitiva la naturaleza comercial o la relevancia de un videojuego. Por ello, se considera necesario validar individualmente estas correspondencias antes de tomar decisiones de inclusión o exclusión.

Los videojuegos icónicos se tratarán como un conjunto complementario y no como parte de la muestra representativa seleccionada mediante criterios de estratificación. Su función será enriquecer determinados análisis y facilitar la interpretación de los resultados mediante ejemplos de títulos reconocibles y relevantes para el mercado.

Esta separación permite conservar el valor analítico y contextual de estos videojuegos sin introducir un sesgo de selección en la muestra principal. En caso de incorporarlos a análisis específicos, se identificarán expresamente como títulos icónicos y se distinguirán de los juegos seleccionados mediante el procedimiento de muestreo.


# Analizamos los juegos con múltiples AppIDs

In [ ]:
# 1. Preparamos el mapping IGDB - Steam
df_mapping_audit = df_steam_map_clean[
    ["game", "uid", "name"]
].copy()

df_mapping_audit = df_mapping_audit.rename(columns={
    "game": "igdb_id",
    "uid": "steam_appid",
    "name": "steam_name"
})

df_mapping_audit["igdb_id"] = pd.to_numeric(
    df_mapping_audit["igdb_id"], errors="coerce"
)

df_mapping_audit["steam_appid"] = pd.to_numeric(
    df_mapping_audit["steam_appid"], errors="coerce"
)

df_mapping_audit = df_mapping_audit.dropna(
    subset=["igdb_id", "steam_appid"]
)

df_mapping_audit["igdb_id"] = df_mapping_audit["igdb_id"].astype(int)
df_mapping_audit["steam_appid"] = df_mapping_audit["steam_appid"].astype(int)


# 2. Identificamos los juegos con más de un AppID
appid_counts = (
    df_mapping_audit.groupby("igdb_id")["steam_appid"]
    .nunique()
)

multiple_igdb_ids = appid_counts[appid_counts > 1].index

print("Juegos con múltiples AppIDs:", len(multiple_igdb_ids))


# 3. Preparamos los metadatos de IGDB
metadata_columns = [
    "id",
    "name",
    "first_release_date",
    "release_year",
    "game_type",
    "genres_names",
    "genres_clean",
    "is_indie"
]

df_igdb_metadata = (
    df_games_clean[metadata_columns]
    .drop_duplicates(subset="id")
    .rename(columns={
        "id": "igdb_id",
        "name": "igdb_name"
    })
)


# 4. Creamos el detalle: una fila por AppID
df_multiple_appids_detail = df_mapping_audit[
    df_mapping_audit["igdb_id"].isin(multiple_igdb_ids)
].merge(
    df_igdb_metadata,
    on="igdb_id",
    how="left"
)

# 5. Añadimos indicadores
sampling_ids = set(
    df_sampling_pool_clean["id"].dropna().astype(int)
)

df_multiple_appids_detail["is_main_game"] = (
    df_multiple_appids_detail["game_type"] == 0
)

df_multiple_appids_detail["in_sampling_pool"] = (
    df_multiple_appids_detail["igdb_id"].isin(sampling_ids)
)

df_multiple_appids_detail["has_steam_mapping"] = True

# 6. Ordenamos para facilitar la revisión
df_multiple_appids_detail = (
    df_multiple_appids_detail
    .sort_values(["igdb_id", "steam_appid"])
    .reset_index(drop=True)
)

display(df_multiple_appids_detail[df_multiple_appids_detail["is_main_game"] == True])

In [ ]:
df_multiple_appids_summary = (
    df_multiple_appids_detail
    .groupby("igdb_id")
    .agg(
        igdb_name=("igdb_name", "first"),
        release_year=("release_year", "first"),
        game_type=("game_type", "first"),
        genres_clean=("genres_clean", "first"),
        is_indie=("is_indie", "first"),
        is_main_game=("is_main_game", "first"),
        in_sampling_pool=("in_sampling_pool", "first"),
        appid_count=("steam_appid", "nunique"),
        steam_appids=("steam_appid", list),
        steam_names=("steam_name", list)
    )
    .reset_index()
    .sort_values(
        ["appid_count", "igdb_name"],
        ascending=[False, True]
    )
)

display(df_multiple_appids_summary)

In [ ]:
print("Total de juegos:", len(df_multiple_appids_summary))

print("\nDistribución de AppIDs por juego:")
display(
    df_multiple_appids_summary["appid_count"]
    .value_counts()
    .sort_index()
)

print("\nMain Games:")
display(
    df_multiple_appids_summary["is_main_game"]
    .value_counts(dropna=False)
)

print("\nPresentes en el sampling pool:")
display(
    df_multiple_appids_summary["in_sampling_pool"]
    .value_counts(dropna=False)
)

In [ ]:
pd.crosstab(
    df_multiple_appids_summary["is_main_game"],
    df_multiple_appids_summary["in_sampling_pool"],
    margins=True
)

Para el muestreo excluiremos los juegos que no sean main_game, que no estén en sampling_pool

In [ ]:
def normalize_name(name):
    if pd.isna(name):
        return ""

    name = unicodedata.normalize("NFKD", str(name))
    name = "".join(
        c for c in name
        if not unicodedata.combining(c)
    )

    name = name.lower()
    name = re.sub(r"[^a-z0-9]+", " ", name)

    return " ".join(name.split())


def classify_steam_name(name):
    name = normalize_name(name)

    patterns = {
        "playtest": r"\bplaytest\b",
        "demo": r"\bdemo\b",
        "prologue": r"\bprologue\b",
        "soundtrack": r"\b(soundtrack|ost)\b",
        "dedicated_server": r"\bdedicated server\b",
        "server": r"\bserver\b",
        "other": None
    }

    for category, pattern in patterns.items():
        if pattern and re.search(pattern, name):
            return category

    return "no_obvious_suffix"


# Expandir AppIDs y nombres
rows = []

for _, row in df_multiple_appids_summary.iterrows():

    for appid, steam_name in zip(
        row["steam_appids"],
        row["steam_names"]
    ):

        rows.append({
            "igdb_id": row["igdb_id"],
            "igdb_name": row["igdb_name"],
            "is_main_game": row["is_main_game"],
            "in_sampling_pool": row["in_sampling_pool"],
            "appid": appid,
            "steam_name": steam_name,
            "name_category": classify_steam_name(steam_name),
            "normalized_igdb": normalize_name(row["igdb_name"]),
            "normalized_steam": normalize_name(steam_name)
        })

df_multiple_appids_expanded = pd.DataFrame(rows)

In [ ]:
df_multiple_appids_expanded[
    "name_category"
].value_counts()

In [ ]:
pd.crosstab(
    df_multiple_appids_expanded["name_category"],
    df_multiple_appids_expanded["in_sampling_pool"],
    margins=True
)

In [ ]:
df_multiple_appids_expanded.loc[
    df_multiple_appids_expanded["name_category"]
    == "no_obvious_suffix",
    [
        "igdb_name",
        "appid",
        "steam_name",
        "in_sampling_pool"
    ]
].sample(
    min(
        30,
        (
            df_multiple_appids_expanded["name_category"]
            == "no_obvious_suffix"
        ).sum()
    ),
    random_state=42
)

In [ ]:
# Resumen de categorías por juego IGDB

df_appid_groups = (
    df_multiple_appids_expanded
    .groupby(
        [
            "igdb_id",
            "igdb_name",
            "in_sampling_pool"
        ],
        as_index=False
    )
    .agg(
        n_no_suffix=(
            "name_category",
            lambda x: (x == "no_obvious_suffix").sum()
        ),
        n_playtest=(
            "name_category",
            lambda x: (x == "playtest").sum()
        ),
        n_demo=(
            "name_category",
            lambda x: (x == "demo").sum()
        ),
        n_prologue=(
            "name_category",
            lambda x: (x == "prologue").sum()
        ),
        n_server=(
            "name_category",
            lambda x: (x == "server").sum()
        ),
        appids=("appid", list),
        steam_names=("steam_name", list),
        categories=("name_category", list)
    )
)

In [ ]:
def classify_appid_group(row):

    if row["n_no_suffix"] == 1 and row["n_playtest"] >= 1:
        return "one_main_plus_playtest"

    if row["n_no_suffix"] >= 2 and row["n_playtest"] >= 1:
        return "multiple_main_plus_playtest"

    if row["n_no_suffix"] == 2:
        return "two_main_candidates"

    if row["n_no_suffix"] == 1 and (
        row["n_demo"] + row["n_prologue"] + row["n_server"] > 0
    ):
        return "one_main_plus_other"

    if row["n_no_suffix"] == 1:
        return "single_main_candidate"

    if row["n_no_suffix"] == 0 and row["n_playtest"] >= 1:
        return "only_playtest"

    if row["n_no_suffix"] == 0:
        return "no_clear_main_candidate"

    return "other_combination"


df_appid_groups["group_pattern"] = (
    df_appid_groups.apply(
        classify_appid_group,
        axis=1
    )
)

In [ ]:
df_appid_groups["group_pattern"].value_counts()

In [ ]:
pd.crosstab(
    df_appid_groups["group_pattern"],
    df_appid_groups["in_sampling_pool"],
    margins=True
)

In [ ]:
df_appid_groups.loc[
    df_appid_groups["n_no_suffix"] >= 2,
    [
        "igdb_id",
        "igdb_name",
        "in_sampling_pool",
        "appids",
        "steam_names",
        "categories"
    ]
].head(30)

In [ ]:
# Juegos in ningún candidato claro
df_appid_groups.loc[
    df_appid_groups["n_no_suffix"] == 0,
    [
        "igdb_id",
        "igdb_name",
        "in_sampling_pool",
        "appids",
        "steam_names",
        "categories"
    ]
]

In [ ]:
# Medir similitud y detectar ediciones

def similarity_score(name1, name2):
    return round(
        SequenceMatcher(
            None,
            normalize_name(name1),
            normalize_name(name2)
        ).ratio() * 100,
        2
    )

In [ ]:
# Candidatos principales de juegos con varias correspondencias
multiple_candidate_ids = set(
    df_appid_groups.loc[
        df_appid_groups["n_no_suffix"] >= 2,
        "igdb_id"
    ]
)

df_main_candidates = df_multiple_appids_expanded.loc[
    df_multiple_appids_expanded["igdb_id"].isin(
        multiple_candidate_ids
    )
    &
    (
        df_multiple_appids_expanded["name_category"]
        == "no_obvious_suffix"
    )
].copy()

df_main_candidates["name_similarity"] = (
    df_main_candidates.apply(
        lambda row: similarity_score(
            row["igdb_name"],
            row["steam_name"]
        ),
        axis=1
    )
)

In [ ]:
# Distribución de candidatos por juego
df_appid_groups[
    "n_no_suffix"
].value_counts().sort_index()

In [ ]:
# Distribución de similitud
pd.cut(
    df_main_candidates["name_similarity"],
    bins=[0, 50, 70, 85, 95, 100],
    labels=[
        "<50",
        "50-70",
        "70-85",
        "85-95",
        "95-100"
    ],
    include_lowest=True
).value_counts().sort_index()

In [ ]:
# Detectar posibles ediciones y versiones
edition_patterns = (
    r"\b(classic|remastered|remake|definitive|"
    r"enhanced|edition|deluxe|complete|"
    r"ultimate|goty|game of the year|"
    r"collection|bundle|single player|"
    r"multiplayer|online|rhcp|"
    r"director.?s cut|hd|redux)\b"
)

df_main_candidates["has_edition_keyword"] = (
    df_main_candidates["steam_name"]
    .apply(
        lambda x: bool(
            re.search(
                edition_patterns,
                normalize_name(x)
            )
        )
    )
)

df_main_candidates["edition_keywords"] = (
    df_main_candidates["steam_name"]
    .apply(
        lambda x: re.findall(
            edition_patterns,
            normalize_name(x)
        )
    )
)

df_main_candidates[
    "has_edition_keyword"
].value_counts()

In [ ]:
sample_ids = (
    df_main_candidates["igdb_id"]
    .drop_duplicates()
    .sample(
        min(30, df_main_candidates["igdb_id"].nunique()),
        random_state=42
    )
)

df_review_candidates = (
    df_main_candidates[
        df_main_candidates["igdb_id"].isin(sample_ids)
    ]
    .sort_values(
        ["igdb_id", "name_similarity"],
        ascending=[True, False]
    )
)

df_review_candidates[
    [
        "igdb_id",
        "igdb_name",
        "appid",
        "steam_name",
        "name_similarity",
        "has_edition_keyword",
        "edition_keywords"
    ]
].to_string(index=False)

In [ ]:
# Indicadores de versiones alternativas o productos auxiliares
review_markers = [
    "beta",
    "alpha",
    "closed alpha",
    "open alpha",
    "teaser",
    "dlc",
    "classic",
    "remastered",
    "remaster",
    "remake",
    "definitive edition",
    "deluxe edition",
    "complete edition",
    "game of the year",
    "goty",
    "single player",
    "multi player",
    "multiplayer",
    "early access"
]


def get_extra_markers(igdb_name, steam_name):

    igdb = f" {normalize_name(igdb_name)} "
    steam = f" {normalize_name(steam_name)} "

    return [
        marker
        for marker in review_markers
        if f" {marker} " in steam
        and f" {marker} " not in igdb
    ]


df_multiple_appids_expanded["extra_markers"] = (
    df_multiple_appids_expanded.apply(
        lambda row: get_extra_markers(
            row["igdb_name"],
            row["steam_name"]
        ),
        axis=1
    )
)

In [ ]:
df_multiple_appids_expanded["name_similarity"] = (
    df_multiple_appids_expanded.apply(
        lambda row: similarity_score(
            row["igdb_name"],
            row["steam_name"]
        ),
        axis=1
    )
)

In [ ]:
decisions = []

for igdb_id, group in df_multiple_appids_expanded.groupby("igdb_id"):

    candidates = group.loc[
        group["name_category"] == "no_obvious_suffix"
    ].copy()

    n_candidates = len(candidates)

    selected_appid = None
    selected_name = None
    decision = "manual_review"
    reason = "ambiguous_matching"
    confidence = "low"

    if n_candidates == 0:

        reason = "no_clear_main_candidate"

    elif n_candidates == 1:

        candidate = candidates.iloc[0]

        if not candidate["extra_markers"]:

            selected_appid = candidate["appid"]
            selected_name = candidate["steam_name"]
            decision = "keep"
            reason = "unique_main_candidate"
            confidence = "high"

        else:

            reason = "unique_candidate_with_version_marker"

    else:

        candidates = candidates.sort_values(
            "name_similarity",
            ascending=False
        )

        best = candidates.iloc[0]
        second = candidates.iloc[1]

        margin = (
            best["name_similarity"]
            - second["name_similarity"]
        )

        if (
            best["name_similarity"] >= 95
            and margin >= 10
            and not best["extra_markers"]
        ):

            selected_appid = best["appid"]
            selected_name = best["steam_name"]
            decision = "keep"
            reason = "unique_high_name_similarity"
            confidence = "high"

        else:

            reason = "multiple_or_ambiguous_candidates"

    decisions.append({
        "igdb_id": igdb_id,
        "igdb_name": group["igdb_name"].iloc[0],
        "n_candidates": n_candidates,
        "selected_appid": selected_appid,
        "selected_name": selected_name,
        "decision": decision,
        "decision_reason": reason,
        "confidence": confidence
    })


df_appid_decisions = pd.DataFrame(decisions)

In [ ]:
df_appid_decisions["decision"].value_counts(dropna=False)

In [ ]:
df_appid_decisions["decision_reason"].value_counts(dropna=False)

In [ ]:
pd.crosstab(
    df_appid_decisions["decision"],
    df_multiple_appids_summary.set_index(
        "igdb_id"
    )["in_sampling_pool"]
)

In [ ]:
df_manual_review = df_appid_decisions.loc[
    df_appid_decisions["decision"] == "manual_review"
].copy()

df_manual_review["decision_reason"].value_counts()

In [ ]:
df_appid_decisions = df_appid_decisions.merge(
    df_multiple_appids_summary[
        ["igdb_id", "in_sampling_pool"]
    ],
    on="igdb_id",
    how="left"
)

In [ ]:
pd.crosstab(
    df_appid_decisions["in_sampling_pool"],
    df_appid_decisions["decision"],
    margins=True
)

In [ ]:
pd.crosstab(
    df_appid_decisions["n_candidates"],
    df_appid_decisions["decision"],
    margins=True
)

In [ ]:
df_manual_review = df_appid_decisions.loc[
    df_appid_decisions["decision"] == "manual_review"
].copy()

print("Juegos pendientes de revisión:", len(df_manual_review))

df_manual_review[
    [
        "igdb_id",
        "igdb_name",
        "n_candidates",
        "decision_reason",
        "in_sampling_pool"
    ]
].head(20)

In [ ]:
df_multiple_appids_expanded.loc[
    df_multiple_appids_expanded["name_category"] == "no_obvious_suffix",
    [
        "igdb_name",
        "steam_name",
        "name_similarity"
    ]
].head(10)

In [ ]:
print("DISTRIBUCIÓN DE DECISIONES")
print(df_appid_decisions["decision"].value_counts())

print("\nMOTIVOS DE DECISIÓN")
print(df_appid_decisions["decision_reason"].value_counts())

print("\nDECISIONES POR NÚMERO DE CANDIDATOS")
print(pd.crosstab(
    df_appid_decisions["n_candidates"],
    df_appid_decisions["decision"],
    margins=True
))

In [ ]:
# Eliminar posibles columnas anteriores de sampling
df_appid_decisions = df_appid_decisions.drop(
    columns=[
        col for col in df_appid_decisions.columns
        if col.startswith("in_sampling_pool")
    ],
    errors="ignore"
)

# Incorporar la columna original
df_appid_decisions = df_appid_decisions.merge(
    df_multiple_appids_summary[
        ["igdb_id", "in_sampling_pool"]
    ].drop_duplicates("igdb_id"),
    on="igdb_id",
    how="left",
    validate="one_to_one"
)

In [ ]:
print(pd.crosstab(
    df_appid_decisions["in_sampling_pool"],
    df_appid_decisions["decision"],
    margins=True
))

In [ ]:
print(df_multiple_appids_summary.columns.tolist())

In [ ]:
# 1. Juegos que requieren revisión
df_manual_review = df_appid_decisions.loc[
    df_appid_decisions["decision"] == "manual_review"
].copy()

# 2. Recuperar todos sus AppIDs candidatos
df_review_candidates = df_multiple_appids_expanded.loc[
    df_multiple_appids_expanded["igdb_id"].isin(
        df_manual_review["igdb_id"]
    )
].copy()

# 3. Incorporar únicamente las columnas adicionales que falten
extra_columns = [
    col for col in [
        "release_year",
        "game_type",
        "in_sampling_pool"
    ]
    if col not in df_review_candidates.columns
]

if extra_columns:
    df_review_candidates = df_review_candidates.merge(
        df_multiple_appids_summary[
            ["igdb_id"] + extra_columns
        ].drop_duplicates("igdb_id"),
        on="igdb_id",
        how="left",
        validate="many_to_one"
    )

# 4. Seleccionar columnas disponibles
columns_review = [
    "igdb_id",
    "igdb_name",
    "release_year",
    "game_type",
    "in_sampling_pool",
    "is_main_game",
    "appid",
    "steam_name",
    "name_category",
    "name_similarity",
    "extra_markers"
]

missing_columns = [
    col for col in columns_review
    if col not in df_review_candidates.columns
]

print("Columnas ausentes:", missing_columns)

display(df_review_candidates.head())

In [ ]:
df_review_candidates.head()

In [ ]:
# Nos quedamos exclusivamente con juegos del sampling pool
df_pool_review = df_review_candidates.loc[
    df_review_candidates["in_sampling_pool"] == True
].copy()

# Contar nombres de Steam distintos por juego IGDB
df_name_counts = (
    df_pool_review
    .groupby("igdb_id")["steam_name"]
    .nunique()
    .rename("n_unique_steam_names")
    .reset_index()
)

# Incorporar el contador
df_pool_review = df_pool_review.merge(
    df_name_counts,
    on="igdb_id",
    how="left"
)

# Separar juegos con nombres diferentes
df_distinct_names = df_pool_review.loc[
    df_pool_review["n_unique_steam_names"] > 1
].copy()

# Separar juegos con nombres idénticos
df_identical_names = df_pool_review.loc[
    df_pool_review["n_unique_steam_names"] == 1
].copy()

print("Juegos con nombres diferentes:",
      df_distinct_names["igdb_id"].nunique())

print("Juegos con nombres idénticos:",
      df_identical_names["igdb_id"].nunique())

In [ ]:
# DataFrames relevantes para la resolución de AppIDs

dfs_to_check = {
    "df_steam_map_clean": df_steam_map_clean,
    "df_steam_map": df_steam_map,
    "df_steam_manifest": df_steam_manifest,
    "df_steam_map_clean_candidates": df_pool_review
}

for name, df in dfs_to_check.items():

    print(f"\n{'='*60}")
    print(name)
    print(f"Dimensiones: {df.shape}")
    print("Columnas:")
    print(df.columns.tolist())

    print("\nPrimeros registros:")
    display(df.head(3))

In [ ]:
# 1. Preparar información de Steam
df_manifest_info = df_steam_manifest[
    [
        "appid",
        "status",
        "reviews_downloaded",
        "reviews_reported",
        "elapsed_seconds"
    ]
].copy()

# Evitar posibles duplicados de AppID
df_manifest_info = df_manifest_info.drop_duplicates(
    subset="appid",
    keep="last"
)

# 2. Incorporar información de extracción
df_candidates_steam = df_pool_review.merge(
    df_manifest_info,
    on="appid",
    how="left",
    indicator=True,
    validate="many_to_one"
)

# 3. Crear indicador de extracción
df_candidates_steam["has_steam_reviews"] = (
    df_candidates_steam["_merge"] == "both"
)

# 4. Resumen general
print("CANDIDATOS")
print("Total:", len(df_candidates_steam))

print("\nESTADO DE EXTRACCIÓN")
print(df_candidates_steam["status"].value_counts(dropna=False))

print("\nPRESENCIA EN MANIFEST")
print(df_candidates_steam["_merge"].value_counts())

print("\nRESEÑAS DESCARGADAS")
print(
    df_candidates_steam["reviews_downloaded"]
    .describe()
)

In [ ]:
df_candidate_comparison = (
    df_candidates_steam
    .sort_values(
        ["igdb_id", "reviews_downloaded"],
        ascending=[True, False]
    )
    [[
        "igdb_id",
        "igdb_name",
        "appid",
        "steam_name",
        "name_similarity",
        "name_category",
        "status",
        "reviews_downloaded",
        "reviews_reported",
        "in_sampling_pool"
    ]]
)

display(df_candidate_comparison.head(30))

In [ ]:
pd.crosstab(
    df_candidates_steam["name_category"],
    df_candidates_steam["status"],
    margins=True,
    dropna=False
)

In [ ]:
# 1. Resumen de extracción por juego IGDB
df_review_status = (
    df_candidates_steam
    .groupby(["igdb_id", "igdb_name"], as_index=False)
    .agg(
        n_candidates=("appid", "nunique"),
        n_extracted=("has_steam_reviews", "sum"),
        n_success=(
            "status",
            lambda x: (x == "success").sum()
        ),
        total_reviews=("reviews_downloaded", "sum"),
        max_reviews=("reviews_downloaded", "max")
    )
)

# 2. Clasificar los juegos según sus extracciones
df_review_status["extraction_pattern"] = "none_extracted"

df_review_status.loc[
    df_review_status["n_success"] == 1,
    "extraction_pattern"
] = "one_success"

df_review_status.loc[
    df_review_status["n_success"] > 1,
    "extraction_pattern"
] = "multiple_success"

# 3. Distribución
print(
    df_review_status["extraction_pattern"]
    .value_counts()
)

# 4. Resumen estadístico
display(
    df_review_status.groupby("extraction_pattern").agg(
        games=("igdb_id", "nunique"),
        candidates=("n_candidates", "sum"),
        avg_reviews=("total_reviews", "mean"),
        max_reviews=("max_reviews", "max")
    )
)

In [ ]:
df_success_cases = df_candidates_steam.loc[
    df_candidates_steam["status"] == "success"
].sort_values(
    ["igdb_id", "reviews_downloaded"],
    ascending=[True, False]
)

display(
    df_success_cases[
        [
            "igdb_id",
            "igdb_name",
            "appid",
            "steam_name",
            "name_similarity",
            "reviews_downloaded",
            "reviews_reported"
        ]
    ]
)

In [ ]:
# Identificar los juegos que tienen un candidato con extracción
success_igdb_ids = df_candidates_steam.loc[
    df_candidates_steam["status"] == "success",
    "igdb_id"
].unique()

# Recuperar TODOS los candidatos de esos juegos
df_success_comparison = df_candidates_steam.loc[
    df_candidates_steam["igdb_id"].isin(success_igdb_ids)
].copy()

# Ordenar por juego y similitud
df_success_comparison = df_success_comparison.sort_values(
    ["igdb_id", "name_similarity"],
    ascending=[True, False]
)

columns = [
    "igdb_id",
    "igdb_name",
    "is_main_game",
    "appid",
    "steam_name",
    "name_category",
    "name_similarity",
    "extra_markers",
    "game_type",
    "status",
    "reviews_downloaded",
    "reviews_reported"
]

display(df_success_comparison[columns].head(20))

In [ ]:
df_success_groups = (
    df_success_comparison
    .groupby(["igdb_id", "igdb_name"])
    .agg(
        n_candidates=("appid", "nunique"),
        n_unique_names=("normalized_steam", "nunique"),
        all_without_markers=(
            "extra_markers",
            lambda x: all(len(markers) == 0 for markers in x)
        ),
        appids=("appid", list)
    )
    .reset_index()
)

df_success_groups["decision"] = "manual_review"

df_success_groups.loc[
    (df_success_groups["n_unique_names"] == 1) &
    df_success_groups["all_without_markers"],
    "decision"
] = "ambiguous_identical_names"

display(df_success_groups)

In [ ]:
review_ids = [
    2369,    # Dark Messiah
    142985,  # Vail
    153430,  # The Palace on the Hill
    325501   # 100 Animals on an Island
]

df_four_cases = (
    df_success_comparison.loc[
        df_success_comparison["igdb_id"].isin(review_ids)
    ]
    .sort_values(["igdb_id", "appid"])
)

columns = [
    "igdb_id",
    "igdb_name",
    "appid",
    "steam_name",
    "name_category",
    "name_similarity",
    "extra_markers",
    "game_type",
    "status",
    "reviews_downloaded",
    "reviews_reported"
]

display(df_four_cases[columns])

In [ ]:
display(
    df_four_cases[
        ["igdb_name", "appid", "steam_name",
         "name_category", "extra_markers"]
    ]
)

In [ ]:
df_success_groups_check = (
    df_success_comparison
    .groupby(["igdb_id", "igdb_name"])
    .agg(
        n_candidates=("appid", "nunique"),
        n_unique_names=("normalized_steam", "nunique"),
        all_without_markers=(
            "appid",
            lambda _: True
        ),
        appids=("appid", list)
    )
    .reset_index()
)

# Comprobar también name_category
marker_check = (
    df_success_comparison
    .assign(
        has_markers=lambda x: (
            (x["name_category"] != "no_obvious_suffix") |
            x["extra_markers"].apply(lambda x: len(x) > 0)
        )
    )
    .groupby("igdb_id")["has_markers"]
    .any()
)

df_success_groups_check["all_without_markers"] = (
    ~df_success_groups_check["igdb_id"].map(marker_check)
)

In [ ]:
selected_decisions = {
    2369: {
        "appid": 2100,
        "reason": "Single Player version selected as the main game"
    },
    142985: {
        "appid": 801550,
        "reason": "Main game selected; Alpha version excluded"
    },
    325501: {
        "appid": 3209400,
        "reason": "Exact title match; alternative has different title"
    }
}

for igdb_id, info in selected_decisions.items():

    mask = df_appid_decisions["igdb_id"] == igdb_id

    df_appid_decisions.loc[mask, "selected_appid"] = info["appid"]
    df_appid_decisions.loc[mask, "decision"] = "keep"
    df_appid_decisions.loc[mask, "decision_reason"] = info["reason"]
    df_appid_decisions.loc[mask, "confidence"] = "high"

In [ ]:
resolved_ids = [
    2369,    # Dark Messiah
    142985,  # Vail
    325501   # 100 Animals on an Island
]

df_identical_pending = df_success_groups.loc[
    (df_success_groups["decision"] == "ambiguous_identical_names") &
    (~df_success_groups["igdb_id"].isin(resolved_ids))
].copy()

print("Juegos pendientes:", len(df_identical_pending))
print("AppIDs implicados:", df_identical_pending["n_candidates"].sum())

In [ ]:
for name, df in {
    #"df_igdb": df_igdb,
    "df_games": df_games,
    "df_steam_map_clean": df_steam_map_clean,
    "df_multiple_appids_expanded": df_multiple_appids_expanded
}.items():
    print(f"\n{name}")
    print(df.columns.tolist())

In [ ]:
pending_ids = df_identical_pending["igdb_id"].unique()

df_identical_candidates = df_candidates_steam.loc[
    df_candidates_steam["igdb_id"].isin(pending_ids)
].copy()

df_identical_candidates = df_identical_candidates.sort_values(
    ["igdb_id", "appid"]
)

display(
    df_identical_candidates[
        [
            "igdb_id",
            "igdb_name",
            "appid",
            "steam_name",
            "name_similarity",
            "name_category",
            "extra_markers",
            "status",
            "reviews_downloaded"
        ]
    ]
)

In [ ]:
df_mapping_extra = df_steam_map_clean[
    ["game", "uid", "year", "url", "external_game_source"]
].copy()

df_mapping_extra["game"] = pd.to_numeric(
    df_mapping_extra["game"],
    errors="coerce"
)

df_mapping_extra["uid"] = pd.to_numeric(
    df_mapping_extra["uid"],
    errors="coerce"
)

df_mapping_extra = df_mapping_extra.rename(
    columns={
        "game": "igdb_id",
        "uid": "appid",
        "year": "steam_mapping_year",
        "url": "steam_mapping_url",
        "external_game_source": "steam_source"
    }
)

# Comprobar que no hay múltiples registros para una misma pareja
print(
    "Parejas duplicadas:",
    df_mapping_extra.duplicated(
        ["igdb_id", "appid"]
    ).sum()
)

df_mapping_extra = df_mapping_extra.drop_duplicates(
    ["igdb_id", "appid"]
)

df_identical_enriched = df_identical_candidates.merge(
    df_mapping_extra,
    on=["igdb_id", "appid"],
    how="left",
    validate="many_to_one"
)

print("Candidatos originales:", len(df_identical_candidates))
print("Candidatos enriquecidos:", len(df_identical_enriched))

In [ ]:
df_year_comparison = (
    df_identical_enriched
    .groupby(["igdb_id", "igdb_name"], as_index=False)
    .agg(
        n_candidates=("appid", "nunique"),
        n_years=("steam_mapping_year", "nunique"),
        min_year=("steam_mapping_year", "min"),
        max_year=("steam_mapping_year", "max"),
        n_sources=("steam_source", "nunique")
    )
)

print("Distribución de años diferentes:")
print(df_year_comparison["n_years"].value_counts())

print("\nDistribución de fuentes diferentes:")
print(df_year_comparison["n_sources"].value_counts())

display(
    df_year_comparison.loc[
        (df_year_comparison["n_years"] > 1) |
        (df_year_comparison["n_sources"] > 1)
    ]
)

In [ ]:
def contains_non_latin(text):
    if pd.isna(text):
        return False

    return bool(
        re.search(
            r"[\u3400-\u9fff\u3040-\u30ff\uac00-\ud7af]",
            str(text)
        )
    )

df_identical_enriched["has_non_latin"] = (
    df_identical_enriched["steam_name"]
    .apply(contains_non_latin)
)

df_suspicious_names = df_identical_enriched.loc[
    df_identical_enriched["has_non_latin"] |
    (df_identical_enriched["name_similarity"] < 90)
].copy()

print(
    "Juegos con títulos que requieren revisión:",
    df_suspicious_names["igdb_id"].nunique()
)

display(
    df_suspicious_names[
        [
            "igdb_id",
            "igdb_name",
            "appid",
            "steam_name",
            "name_similarity",
            "steam_mapping_year",
            "status"
        ]
    ]
)

In [ ]:
def normalize_name_unicode(text):
    if pd.isna(text):
        return ""

    text = unicodedata.normalize("NFKC", str(text))
    text = text.casefold()

    # Sustituir puntuación por espacios, conservando letras y números
    text = "".join(
        char if char.isalnum() else " "
        for char in text
    )

    text = re.sub(r"\s+", " ", text).strip()

    return text


df_identical_enriched["normalized_steam_unicode"] = (
    df_identical_enriched["steam_name"]
    .apply(normalize_name_unicode)
)

In [ ]:
df_unicode_comparison = (
    df_identical_enriched
    .groupby(["igdb_id", "igdb_name"], as_index=False)
    .agg(
        n_candidates=("appid", "nunique"),
        n_unique_names_old=("normalized_steam", "nunique"),
        n_unique_names_unicode=(
            "normalized_steam_unicode", "nunique"
        )
    )
)

df_unicode_comparison["normalization_changed"] = (
    df_unicode_comparison["n_unique_names_unicode"] >
    df_unicode_comparison["n_unique_names_old"]
)

print(
    df_unicode_comparison["normalization_changed"]
    .value_counts()
)

display(
    df_unicode_comparison.loc[
        df_unicode_comparison["normalization_changed"]
    ]
)

In [ ]:
# Año de lanzamiento según IGDB
df_igdb_years = (
    df_games[["id", "release_year"]]
    .rename(columns={
        "id": "igdb_id",
        "release_year": "igdb_release_year"
    })
    .drop_duplicates("igdb_id")
)

# Evitar columnas duplicadas si ejecutamos la celda varias veces
df_year_audit = df_identical_enriched.drop(
    columns=["igdb_release_year"],
    errors="ignore"
).copy()

# Incorporar el año desde df_games
df_year_audit = df_year_audit.merge(
    df_igdb_years,
    on="igdb_id",
    how="left",
    validate="many_to_one"
)

# Calcular diferencia entre años
df_year_audit["year_difference"] = (
    pd.to_numeric(
        df_year_audit["steam_mapping_year"],
        errors="coerce"
    )
    -
    pd.to_numeric(
        df_year_audit["igdb_release_year"],
        errors="coerce"
    )
)

# Juegos con años diferentes entre sus AppIDs
different_year_ids = df_year_comparison.loc[
    df_year_comparison["n_years"] > 1,
    "igdb_id"
]

display(
    df_year_audit.loc[
        df_year_audit["igdb_id"].isin(different_year_ids),
        [
            "igdb_id",
            "igdb_name",
            "appid",
            "steam_name",
            "igdb_release_year",
            "steam_mapping_year",
            "year_difference",
            "status"
        ]
    ].sort_values(["igdb_id", "steam_mapping_year"])
)

In [ ]:
mask = df_appid_decisions["igdb_id"] == 89613

df_appid_decisions.loc[mask, "selected_appid"] = 813300
df_appid_decisions.loc[mask, "decision"] = "keep"
df_appid_decisions.loc[mask, "decision_reason"] = (
    "Main game selected; alternative identified as soundtrack"
)
df_appid_decisions.loc[mask, "confidence"] = "high"

In [ ]:
# Resumen de los 43 grupos
df_pending_audit = (
    df_identical_enriched
    .groupby(["igdb_id", "igdb_name"], as_index=False)
    .agg(
        n_candidates=("appid", "nunique"),
        n_unique_names_unicode=(
            "normalized_steam_unicode", "nunique"
        ),
        n_mapping_years=("steam_mapping_year", "nunique"),
        n_extracted=(
            "status",
            lambda x: (x == "success").sum()
        ),
        appids=("appid", list)
    )
)

# Señales de revisión
df_pending_audit["different_names"] = (
    df_pending_audit["n_unique_names_unicode"] > 1
)

df_pending_audit["different_years"] = (
    df_pending_audit["n_mapping_years"] > 1
)

# Categoría de auditoría
df_pending_audit["audit_category"] = "identical_no_year_difference"

df_pending_audit.loc[
    df_pending_audit["different_years"],
    "audit_category"
] = "identical_with_year_difference"

df_pending_audit.loc[
    df_pending_audit["different_names"],
    "audit_category"
] = "different_names"

print(df_pending_audit["audit_category"].value_counts())

display(
    df_pending_audit.sort_values(
        ["audit_category", "igdb_id"]
    )
)

In [ ]:
# Comprobar que tenemos una única decisión por juego
assert df_appid_decisions["igdb_id"].is_unique

# Crear una copia para no modificar el original
df_appid_decisions_v2 = df_appid_decisions.copy()

# Eliminar columnas de auditoría previas si se vuelve a ejecutar
audit_columns = [
    "audit_category",
    "n_unique_names_unicode",
    "n_mapping_years"
]

df_appid_decisions_v2 = df_appid_decisions_v2.drop(
    columns=audit_columns,
    errors="ignore"
)

# Incorporar los resultados de los 43 juegos
df_appid_decisions_v2 = df_appid_decisions_v2.merge(
    df_pending_audit[
        ["igdb_id"] + audit_columns
    ],
    on="igdb_id",
    how="left",
    validate="one_to_one"
)

print("Decisiones originales:", len(df_appid_decisions))
print("Decisiones actualizadas:", len(df_appid_decisions_v2))

assert len(df_appid_decisions_v2) == len(df_appid_decisions)

In [ ]:
print("ESTADO GLOBAL DE DECISIONES")
print(
    df_appid_decisions_v2["decision"]
    .value_counts(dropna=False)
)

print("\nESTADO DENTRO DEL SAMPLING POOL")
print(
    df_appid_decisions_v2.loc[
        df_appid_decisions_v2["in_sampling_pool"] == True,
        "decision"
    ].value_counts(dropna=False)
)

print("\nCASOS PENDIENTES POR CATEGORÍA DE AUDITORÍA")
print(
    df_appid_decisions_v2.loc[
        df_appid_decisions_v2["decision"] == "manual_review",
        "audit_category"
    ].value_counts(dropna=False)
)

In [ ]:
# Juegos pendientes dentro del sampling pool
df_pending_pool = df_appid_decisions_v2.loc[
    (df_appid_decisions_v2["decision"] == "manual_review") &
    (df_appid_decisions_v2["in_sampling_pool"] == True)
].copy()

# Recuperar todos sus candidatos
df_pending_candidates = df_multiple_appids_expanded.loc[
    df_multiple_appids_expanded["igdb_id"].isin(
        df_pending_pool["igdb_id"]
    )
].copy()

print("Juegos pendientes:", len(df_pending_pool))

print(
    "Juegos con candidatos:",
    df_pending_candidates["igdb_id"].nunique()
)

print("AppIDs candidatos:", len(df_pending_candidates))

In [ ]:
# Comprobar si hay marcadores adicionales
def has_extra_markers(markers):
    if isinstance(markers, (list, tuple, set)):
        return len(markers) > 0
    return False

df_pending_candidates["has_markers"] = (
    df_pending_candidates["extra_markers"]
    .apply(has_extra_markers)
    |
    df_pending_candidates["name_category"].ne(
        "no_obvious_suffix"
    )
)

# Aplicar normalización Unicode
df_pending_candidates["normalized_steam_unicode"] = (
    df_pending_candidates["steam_name"]
    .apply(normalize_name_unicode)
)

# Resumen por juego
df_pending_summary = (
    df_pending_candidates
    .groupby("igdb_id", as_index=False)
    .agg(
        igdb_name=("igdb_name", "first"),
        n_candidates=("appid", "nunique"),
        n_unique_names=("normalized_steam_unicode", "nunique"),
        any_markers=("has_markers", "any"),
        max_similarity=("name_similarity", "max"),
        min_similarity=("name_similarity", "min")
    )
)

print("Juegos resumidos:", len(df_pending_summary))

print("\nDistribución de candidatos:")
print(df_pending_summary["n_candidates"].value_counts())

print("\nJuegos con marcadores:")
print(df_pending_summary["any_markers"].value_counts())

print("\nJuegos con nombres diferentes:")
print(
    (df_pending_summary["n_unique_names"] > 1)
    .value_counts()
)

In [ ]:
df_pending_summary["review_group"] = np.select(
    [
        df_pending_summary["any_markers"],
        df_pending_summary["n_unique_names"] > 1
    ],
    [
        "A_markers",
        "B_different_names"
    ],
    default="C_equivalent_names"
)

print(
    df_pending_summary["review_group"]
    .value_counts()
)

display(
    df_pending_summary.groupby("review_group").agg(
        n_games=("igdb_id", "nunique"),
        n_candidates=("n_candidates", "sum"),
        avg_candidates=("n_candidates", "mean")
    )
)

In [ ]:
marker_ids = df_pending_summary.loc[
    df_pending_summary["review_group"] == "A_markers",
    "igdb_id"
]

df_marker_candidates = df_pending_candidates.loc[
    df_pending_candidates["igdb_id"].isin(marker_ids)
].copy()

display(
    df_marker_candidates[
        [
            "igdb_id",
            "igdb_name",
            "appid",
            "steam_name",
            "name_category",
            "extra_markers",
            "name_similarity"
        ]
    ].sort_values(["igdb_id", "appid"])
)

In [ ]:
# Comprobaremos que los ocho juegos siguen pendientes y que todos los AppIDs elegidos pertenecen a sus candidatos
selected_main_games = {
    545: 42700,
    9509: 311210,
    37390: 600280,
    95637: 813670,
    114058: 1011670,
    130261: 1228500,
    195231: 1625340,
    211944: 1866980
}

for igdb_id, appid in selected_main_games.items():
    candidates = df_pending_candidates.loc[
        df_pending_candidates["igdb_id"] == igdb_id,
        "appid"
    ]

    assert appid in candidates.values, (
        f"AppID {appid} no encontrado para IGDB {igdb_id}"
    )

    current = df_appid_decisions_v2.loc[
        df_appid_decisions_v2["igdb_id"] == igdb_id,
        "decision"
    ]

    assert len(current) == 1 and current.iloc[0] == "manual_review", (
        f"IGDB {igdb_id} no está pendiente"
    )

print("Validación completada: 8 selecciones correctas")

In [ ]:
for igdb_id, appid in selected_main_games.items():

    mask = df_appid_decisions_v2["igdb_id"] == igdb_id

    df_appid_decisions_v2.loc[mask, "selected_appid"] = appid
    df_appid_decisions_v2.loc[mask, "decision"] = "keep"
    df_appid_decisions_v2.loc[mask, "decision_reason"] = (
        "Main game selected over explicitly marked "
        "alternative version or supplementary product"
    )
    df_appid_decisions_v2.loc[mask, "confidence"] = "high"

print(
    df_appid_decisions_v2.loc[
        df_appid_decisions_v2["in_sampling_pool"] == True,
        "decision"
    ].value_counts()
)

In [ ]:
# Identificar los juegos del grupo A que siguen pendientes
pending_marker_ids = df_appid_decisions_v2.loc[
    (df_appid_decisions_v2["decision"] == "manual_review") &
    (df_appid_decisions_v2["igdb_id"].isin(
        df_marker_candidates["igdb_id"]
    )),
    "igdb_id"
]

# Recuperar sus candidatos y añadir información del mapping
df_marker_review = df_marker_candidates.loc[
    df_marker_candidates["igdb_id"].isin(pending_marker_ids)
].copy()

df_marker_review = df_marker_review.merge(
    df_mapping_extra[
        ["igdb_id", "appid", "steam_mapping_year"]
    ],
    on=["igdb_id", "appid"],
    how="left",
    validate="many_to_one"
)

# Añadir año y tipo de juego de IGDB
df_marker_review = df_marker_review.merge(
    df_games[
        ["id", "release_year", "game_type_name"]
    ].rename(columns={
        "id": "igdb_id",
        "release_year": "igdb_release_year"
    }).drop_duplicates("igdb_id"),
    on="igdb_id",
    how="left",
    validate="many_to_one"
)

print("Juegos pendientes:", df_marker_review["igdb_id"].nunique())
print("AppIDs candidatos:", len(df_marker_review))

display(
    df_marker_review[
        [
            "igdb_id",
            "igdb_name",
            "igdb_release_year",
            "game_type_name",
            "appid",
            "steam_name",
            "steam_mapping_year",
            "name_category",
            "extra_markers"
        ]
    ].sort_values(["igdb_id", "appid"])
)

In [ ]:
additional_decisions = {
    36605: {
        "appid": 349510,
        "reason": "Early Access game selected over Playtest",
        "confidence": "high"
    },
    114684: {
        "appid": 677620,
        "reason": (
            "Original 2019 release selected over 2022 Playtest; "
            "mapping title includes Alpha"
        ),
        "confidence": "medium"
    }
}

for igdb_id, info in additional_decisions.items():

    mask = df_appid_decisions_v2["igdb_id"] == igdb_id

    assert mask.sum() == 1
    assert df_appid_decisions_v2.loc[
        mask, "decision"
    ].iloc[0] == "manual_review"

    assert info["appid"] in df_marker_review.loc[
        df_marker_review["igdb_id"] == igdb_id,
        "appid"
    ].values

print("Validación completada")

In [ ]:
for igdb_id, info in additional_decisions.items():

    mask = df_appid_decisions_v2["igdb_id"] == igdb_id

    df_appid_decisions_v2.loc[mask, "selected_appid"] = info["appid"]
    df_appid_decisions_v2.loc[mask, "decision"] = "keep"
    df_appid_decisions_v2.loc[mask, "decision_reason"] = info["reason"]
    df_appid_decisions_v2.loc[mask, "confidence"] = info["confidence"]

print(
    df_appid_decisions_v2.loc[
        df_appid_decisions_v2["in_sampling_pool"],
        "decision"
    ].value_counts()
)

In [ ]:
# Juegos del grupo B que siguen pendientes
group_b_ids = df_pending_summary.loc[
    df_pending_summary["review_group"] == "B_different_names",
    "igdb_id"
]

pending_ids = df_appid_decisions_v2.loc[
    df_appid_decisions_v2["decision"] == "manual_review",
    "igdb_id"
]

group_b_pending_ids = group_b_ids[
    group_b_ids.isin(pending_ids)
]

df_group_b = df_pending_candidates.loc[
    df_pending_candidates["igdb_id"].isin(group_b_pending_ids)
].copy()

print("Juegos pendientes del grupo B:", df_group_b["igdb_id"].nunique())
print("AppIDs candidatos:", len(df_group_b))

In [ ]:
df_group_b_summary = df_pending_summary.loc[
    df_pending_summary["igdb_id"].isin(group_b_pending_ids)
].copy()

# Diferencia entre el candidato más similar y el menos similar
df_group_b_summary["similarity_gap"] = (
    df_group_b_summary["max_similarity"] -
    df_group_b_summary["min_similarity"]
)

print("DISTRIBUCIÓN DE SIMILITUD MÁXIMA")

print(
    pd.cut(
        df_group_b_summary["max_similarity"],
        bins=[-0.01, 50, 80, 95, 100],
        labels=[
            "0-50",
            "50-80",
            "80-95",
            "95-100"
        ],
        include_lowest=True
    ).value_counts().sort_index()
)

print("\nDIFERENCIA ENTRE CANDIDATOS")

print(
    pd.cut(
        df_group_b_summary["similarity_gap"],
        bins=[-0.01, 5, 15, 30, 100],
        labels=[
            "0-5",
            "5-15",
            "15-30",
            "30-100"
        ],
        include_lowest=True
    ).value_counts().sort_index()
)

In [ ]:
priority_ids = df_group_b_summary.loc[
    df_group_b_summary["similarity_gap"] >= 15,
    "igdb_id"
]

df_group_b_priority = df_group_b.loc[
    df_group_b["igdb_id"].isin(priority_ids)
].copy()

print(
    "Juegos prioritarios:",
    df_group_b_priority["igdb_id"].nunique()
)

display(
    df_group_b_priority[
        [
            "igdb_id",
            "igdb_name",
            "appid",
            "steam_name",
            "name_similarity",
            "name_category",
            "extra_markers"
        ]
    ].sort_values(
        ["igdb_id", "name_similarity"],
        ascending=[True, False]
    )
)

In [ ]:
# Enriquecer los candidatos prioritarios
df_group_b_priority_enriched = df_group_b_priority.merge(
    df_mapping_extra[
        ["igdb_id", "appid", "steam_mapping_year"]
    ],
    on=["igdb_id", "appid"],
    how="left",
    validate="many_to_one"
)

df_group_b_priority_enriched = df_group_b_priority_enriched.merge(
    df_games[
        ["id", "release_year", "game_type_name"]
    ].rename(columns={
        "id": "igdb_id",
        "release_year": "igdb_release_year"
    }).drop_duplicates("igdb_id"),
    on="igdb_id",
    how="left",
    validate="many_to_one"
)

print(
    "Juegos:",
    df_group_b_priority_enriched["igdb_id"].nunique()
)
print(
    "Candidatos:",
    len(df_group_b_priority_enriched)
)

# Evitar truncamiento de títulos
with pd.option_context("display.max_colwidth", None):
    display(
        df_group_b_priority_enriched[
            [
                "igdb_id",
                "igdb_name",
                "igdb_release_year",
                "appid",
                "steam_name",
                "steam_mapping_year",
                "name_similarity"
            ]
        ].sort_values(["igdb_id", "appid"])
    )

In [ ]:
igdb_id = 128770
selected_appid = 1194250

mask = df_appid_decisions_v2["igdb_id"] == igdb_id

assert mask.sum() == 1

assert (
    df_appid_decisions_v2.loc[mask, "decision"].iloc[0]
    == "manual_review"
)

assert selected_appid in df_group_b.loc[
    df_group_b["igdb_id"] == igdb_id,
    "appid"
].values

print("Validación completada")

In [ ]:
df_appid_decisions_v2.loc[mask, "selected_appid"] = selected_appid
df_appid_decisions_v2.loc[mask, "selected_name"] = "Screaming Chicken！"
df_appid_decisions_v2.loc[mask, "decision"] = "keep"

df_appid_decisions_v2.loc[mask, "decision_reason"] = (
    "Main game selected over explicitly labelled Play Test"
)

df_appid_decisions_v2.loc[mask, "confidence"] = "high"

# Construcción del dataset de juegos candidatos definitivo

In [ ]:
# 1. Comprobar dimensiones
datasets = {
    "df_games": df_games,
    "df_steam_map_clean": df_steam_map_clean,
    "df_appid_decisions_v2": df_appid_decisions_v2
}

for name, df in datasets.items():
    print(f"\n{name}")
    print("Filas:", len(df))
    print("Columnas:", df.columns.tolist())


# 2. Comprobar identificadores únicos
print("\n--- IDENTIFICADORES ---")

print(
    "IGDB IDs únicos:",
    df_games["id"].nunique()
)

print(
    "Juegos IGDB con Steam mapping:",
    df_steam_map_clean["game"].nunique()
)

print(
    "Juegos con decisiones:",
    df_appid_decisions_v2["igdb_id"].nunique()
)


# 3. Comprobar decisiones
print("\n--- DECISIONES ---")

display(
    df_appid_decisions_v2.groupby(
        ["in_sampling_pool", "decision"],
        dropna=False
    ).size().reset_index(name="n_games")
)


# 4. Comprobar exclusiones por fecha
print("\n--- FECHAS ---")

print(
    "Sin release_year:",
    df_games["release_year"].isna().sum()
)

print(
    "Release year >= 2027:",
    (df_games["release_year"] >= 2027).sum()
)

print(
    "Release year válido:",
    (
        df_games["release_year"].notna() &
        (df_games["release_year"] < 2027)
    ).sum()
)

In [ ]:
# Trabajamos con una copia del mapping
df_mapping_check = df_steam_map_clean.copy()

df_mapping_check["game"] = pd.to_numeric(
    df_mapping_check["game"],
    errors="coerce"
)

df_mapping_check["uid"] = pd.to_numeric(
    df_mapping_check["uid"],
    errors="coerce"
)

# Número de AppIDs distintos por juego IGDB
df_appid_counts = (
    df_mapping_check
    .dropna(subset=["game", "uid"])
    .groupby("game")["uid"]
    .nunique()
    .reset_index(name="n_appids")
)

single_ids = set(
    df_appid_counts.loc[
        df_appid_counts["n_appids"] == 1, "game"
    ]
)

multiple_ids = set(
    df_appid_counts.loc[
        df_appid_counts["n_appids"] > 1, "game"
    ]
)

decision_ids = set(
    df_appid_decisions_v2["igdb_id"].dropna()
)

print("--- COBERTURA DEL MAPPING ---")

print("Juegos con un AppID:", len(single_ids))
print("Juegos con varios AppIDs:", len(multiple_ids))
print("Juegos con decisiones:", len(decision_ids))

print(
    "Múltiples sin decisión:",
    len(multiple_ids - decision_ids)
)

print(
    "Decisiones sin múltiples AppIDs actuales:",
    len(decision_ids - multiple_ids)
)

print("\n--- CALIDAD DEL MAPPING ---")

print(
    "Filas con IGDB ID nulo:",
    df_mapping_check["game"].isna().sum()
)

print(
    "Filas con AppID nulo:",
    df_mapping_check["uid"].isna().sum()
)

print(
    "Parejas IGDB-AppID duplicadas:",
    df_mapping_check.duplicated(
        ["game", "uid"]
    ).sum()
)

print("\n--- CASOS SIN COBERTURA ---")

display(
    df_appid_counts.loc[
        df_appid_counts["game"].isin(
            multiple_ids - decision_ids
        )
    ].head(20)
)

In [ ]:
# Copia del mapping
df_mapping_base = df_mapping_check.dropna(
    subset=["game", "uid"]
).copy()

df_mapping_base = df_mapping_base.drop_duplicates(
    subset=["game", "uid"]
)

df_mapping_base["game"] = df_mapping_base["game"].astype("int64")
df_mapping_base["uid"] = df_mapping_base["uid"].astype("int64")

# Juegos con un único AppID
df_mapping_single = df_mapping_base.loc[
    df_mapping_base["game"].isin(single_ids)
].copy()

df_mapping_single["mapping_source"] = "single_appid"

# Juegos con múltiples AppIDs y decisión keep
df_decisions_keep = df_appid_decisions_v2.loc[
    df_appid_decisions_v2["decision"] == "keep",
    ["igdb_id", "selected_appid"]
].copy()

df_decisions_keep["igdb_id"] = pd.to_numeric(
    df_decisions_keep["igdb_id"],
    errors="coerce"
)

df_decisions_keep["selected_appid"] = pd.to_numeric(
    df_decisions_keep["selected_appid"],
    errors="coerce"
)

df_mapping_multiple_keep = df_mapping_base.merge(
    df_decisions_keep,
    left_on=["game", "uid"],
    right_on=["igdb_id", "selected_appid"],
    how="inner",
    validate="one_to_one"
)

df_mapping_multiple_keep["mapping_source"] = "multiple_appids_resolved"

# Unificar
df_mapping_final = pd.concat(
    [
        df_mapping_single,
        df_mapping_multiple_keep
    ],
    ignore_index=True
)

print("Juegos con AppID único:", df_mapping_single["game"].nunique())
print("Juegos múltiples resueltos:", df_mapping_multiple_keep["game"].nunique())
print("Mapping final:", df_mapping_final["game"].nunique())

print(
    "IGDB duplicados:",
    df_mapping_final["game"].duplicated().sum()
)

In [ ]:
df_candidates = df_mapping_final.merge(
    df_games,
    left_on="game",
    right_on="id",
    how="inner",
    validate="one_to_one",
    suffixes=("_steam", "_igdb")
)

print("Candidatos antes de filtros:", len(df_candidates))

In [ ]:
df_candidates = df_candidates.loc[
    df_candidates["release_year"].notna() &
    (df_candidates["release_year"] < 2027)
].copy()

print("Candidatos con fecha válida:", len(df_candidates))

In [ ]:
print("Nulos en genres:", df_candidates["genres"].isna().sum())

print("\nEjemplos de valores:")
display(
    df_candidates["genres"].head(15)
)

print("\nTipos de datos:")
print(
    df_candidates["genres"].apply(
        lambda x: type(x).__name__
    ).value_counts()
)

In [ ]:
# Comprobar si el juego tiene al menos un género
def has_valid_genre(genres):
    return (
        isinstance(genres, list)
        and len(genres) > 0
    )

df_candidates["has_valid_genre"] = (
    df_candidates["genres"].apply(has_valid_genre)
)

print("--- FILTRO DE GÉNEROS ---")

print("Candidatos antes:", len(df_candidates))

print(
    "Con género válido:",
    df_candidates["has_valid_genre"].sum()
)

print(
    "Sin género válido:",
    (~df_candidates["has_valid_genre"]).sum()
)

# Aplicar filtro
df_candidates_final = df_candidates.loc[
    df_candidates["has_valid_genre"]
].copy()

print(
    "\nCandidatos después del filtro:",
    len(df_candidates_final)
)

In [ ]:
print("--- VALIDACIÓN FINAL ---")

print("Juegos:", len(df_candidates_final))

print(
    "IGDB IDs únicos:",
    df_candidates_final["id_igdb"].nunique()
)

print(
    "AppIDs únicos:",
    df_candidates_final["uid"].nunique()
)

print(
    "IGDB IDs duplicados:",
    df_candidates_final["id_igdb"].duplicated().sum()
)

print(
    "AppIDs duplicados:",
    df_candidates_final["uid"].duplicated().sum()
)

print(
    "IGDB IDs inconsistentes con mapping:",
    (
        df_candidates_final["id_igdb"] !=
        df_candidates_final["game"]
    ).sum()
)

print(
    "AppIDs nulos:",
    df_candidates_final["uid"].isna().sum()
)

print(
    "Fechas nulas:",
    df_candidates_final["release_year"].isna().sum()
)

print(
    "Juegos desde 2027:",
    (df_candidates_final["release_year"] >= 2027).sum()
)

In [ ]:
df_candidates_final = df_candidates_final.drop(
    columns=[
        "igdb_id",
        "selected_appid",
        "id_steam",
        "game",
        "has_valid_genre"
    ],
    errors="ignore"
)

df_candidates_final = df_candidates_final.rename(
    columns={
        "id_igdb": "igdb_id",
        "uid": "appid",
        "name_igdb": "game_name",
        "name_steam": "steam_name"
    }
)

assert df_candidates_final["igdb_id"].is_unique
assert df_candidates_final["appid"].is_unique

print("Dataset preparado:", df_candidates_final.shape)

In [ ]:
OUTPUT_DIR = Path("sampling")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df_candidates_final.to_pickle(
    OUTPUT_DIR / "candidates_final.pkl"
)

print("Dataset guardado correctamente")

# Selección de una muestra estratificada multietiqueta

In [ ]:
# Distribución por año
df_year_distribution = (
    df_candidates_final
    .groupby("release_year")
    .size()
    .reset_index(name="n_games")
    .sort_values("release_year")
)

print("--- DISTRIBUCIÓN TEMPORAL ---")

print(
    "Año mínimo:",
    df_candidates_final["release_year"].min()
)

print(
    "Año máximo:",
    df_candidates_final["release_year"].max()
)

print(
    "Mediana:",
    df_candidates_final["release_year"].median()
)

print("\nTop 15 años con más videojuegos:")

display(
    df_year_distribution
    .sort_values("n_games", ascending=False)
    .head(15)
)

# Gráfico
plt.figure(figsize=(14, 5))

plt.bar(
    df_year_distribution["release_year"],
    df_year_distribution["n_games"]
)

plt.title("Distribución de videojuegos candidatos por año")
plt.xlabel("Año de lanzamiento")
plt.ylabel("Número de videojuegos")

plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
df_type_distribution = (
    df_candidates_final["game_type_name"]
    .fillna("Unknown")
    .value_counts()
    .reset_index()
)

df_type_distribution.columns = [
    "game_type_name",
    "n_games"
]

df_type_distribution["percentage"] = (
    df_type_distribution["n_games"]
    / len(df_candidates_final)
    * 100
).round(2)

print("--- DISTRIBUCIÓN POR TIPO DE JUEGO ---")

display(df_type_distribution)

# Gráfico
plt.figure(figsize=(11, 5))

plt.barh(
    df_type_distribution["game_type_name"],
    df_type_distribution["n_games"]
)

plt.gca().invert_yaxis()

plt.title("Distribución por tipo de videojuego")
plt.xlabel("Número de videojuegos")
plt.ylabel("Tipo")

plt.tight_layout()
plt.show()

In [ ]:
# Expandir listas de géneros
df_genres_exploded = (
    df_candidates_final[
        ["igdb_id", "genres"]
    ]
    .explode("genres")
    .dropna(subset=["genres"])
)

# Contar videojuegos únicos por género
df_genre_distribution = (
    df_genres_exploded
    .groupby("genres")["igdb_id"]
    .nunique()
    .reset_index(name="n_games")
    .sort_values("n_games", ascending=False)
)

df_genre_distribution["percentage"] = (
    df_genre_distribution["n_games"]
    / len(df_candidates_final)
    * 100
).round(2)

print("--- DISTRIBUCIÓN POR GÉNEROS ---")

display(df_genre_distribution)

# Gráfico de los 15 géneros más frecuentes
top_genres = df_genre_distribution.head(15)

plt.figure(figsize=(11, 6))

plt.barh(
    top_genres["genres"].astype(str),
    top_genres["n_games"]
)

plt.gca().invert_yaxis()

plt.title("Top 15 géneros de videojuegos")
plt.xlabel("Número de videojuegos")
plt.ylabel("IGDB Genre ID")

plt.tight_layout()
plt.show()

In [ ]:
columns_to_check = [
    "platforms",
    "game_modes",
    "themes",
    "player_perspectives",
    "involved_companies",
    "rating",
    "rating_count",
    "total_rating",
    "total_rating_count",
    "summary"
]

def has_data(value):
    if isinstance(value, (list, dict)):
        return len(value) > 0
    return pd.notna(value) and str(value).strip() != ""

df_coverage = pd.DataFrame({
    "column": columns_to_check,
    "n_available": [
        df_candidates_final[col].apply(has_data).sum()
        for col in columns_to_check
    ]
})

df_coverage["coverage_pct"] = (
    df_coverage["n_available"]
    / len(df_candidates_final)
    * 100
).round(2)

display(
    df_coverage.sort_values(
        "coverage_pct",
        ascending=False
    )
)

In [ ]:
# Trabajamos sobre una copia
df_sampling_pool = df_candidates_final.loc[
    df_candidates_final["game_type_name"] == "main_game"
].copy()

print("Main games iniciales:", len(df_sampling_pool))

# Separar Indie del resto de géneros
INDIE_GENRE_ID = 32

df_sampling_pool["is_indie"] = (
    df_sampling_pool["genres"].apply(
        lambda genres: INDIE_GENRE_ID in genres
    )
)

# Eliminar Indie de la lista de géneros
df_sampling_pool["genres"] = (
    df_sampling_pool["genres"].apply(
        lambda genres: [
            genre for genre in genres
            if genre != INDIE_GENRE_ID
        ]
    )
)

# Identificar juegos sin géneros después de eliminar Indie
df_sampling_pool["has_game_genre"] = (
    df_sampling_pool["genres"].apply(
        lambda genres: len(genres) > 0
    )
)

print(
    "Indie:",
    df_sampling_pool["is_indie"].sum()
)

print(
    "Sin género de juego:",
    (~df_sampling_pool["has_game_genre"]).sum()
)

# Conservar únicamente juegos con género identificado
df_sampling_pool = df_sampling_pool.loc[
    df_sampling_pool["has_game_genre"]
].copy()

df_sampling_pool = df_sampling_pool.drop(
    columns=["has_game_genre"]
)

print(
    "\nUniverso final de muestreo:",
    len(df_sampling_pool)
)

In [ ]:
print("--- VALIDACIÓN DEL SAMPLING POOL ---")

print("Juegos:", len(df_sampling_pool))

print(
    "IGDB IDs únicos:",
    df_sampling_pool["igdb_id"].nunique()
)

print(
    "AppIDs únicos:",
    df_sampling_pool["appid"].nunique()
)

print(
    "Todos main_game:",
    df_sampling_pool["game_type_name"].eq("main_game").all()
)

print(
    "Juegos con Indie todavía en genres:",
    df_sampling_pool["genres"].apply(
        lambda genres: 32 in genres
    ).sum()
)

print(
    "Porcentaje indie:",
    round(df_sampling_pool["is_indie"].mean() * 100, 2)
)

print(
    "Periodo:",
    df_sampling_pool["release_year"].min(),
    "-",
    df_sampling_pool["release_year"].max()
)

Periodo	y       Justificación

Antes de 2000	--> Videojuegos clásicos

2000–2009	    --> Primera etapa de distribución digital

2010–2014	    --> Expansión del mercado digital

2015–2019	    --> Consolidación del mercado indie y Steam

2020–2022	    --> Mercado reciente

2023–2026	    --> Lanzamientos más recientes

In [ ]:
year_bins = [
    1969, 1999, 2009, 2014,
    2019, 2022, 2026
]

year_labels = [
    "Antes de 2000",
    "2000-2009",
    "2010-2014",
    "2015-2019",
    "2020-2022",
    "2023-2026"
]

df_sampling_pool["release_period"] = pd.cut(
    df_sampling_pool["release_year"],
    bins=year_bins,
    labels=year_labels
)

df_period_distribution = (
    df_sampling_pool["release_period"]
    .value_counts(sort=False)
    .reset_index()
)

df_period_distribution.columns = [
    "release_period",
    "n_games"
]

df_period_distribution["percentage"] = (
    df_period_distribution["n_games"]
    / len(df_sampling_pool)
    * 100
).round(2)

print("--- DISTRIBUCIÓN POR PERIODOS ---")

display(df_period_distribution)

print(
    "\nJuegos sin periodo:",
    df_sampling_pool["release_period"].isna().sum()
)

In [ ]:
# Distribución de géneros del universo de muestreo
df_genres_pool = (
    df_sampling_pool[["igdb_id", "genres"]]
    .explode("genres")
    .dropna(subset=["genres"])
)

df_genre_distribution_pool = (
    df_genres_pool
    .groupby("genres")["igdb_id"]
    .nunique()
    .reset_index(name="n_games")
    .sort_values("n_games", ascending=False)
)

df_genre_distribution_pool["percentage"] = (
    df_genre_distribution_pool["n_games"]
    / len(df_sampling_pool)
    * 100
).round(2)

print("--- GÉNEROS EN MAIN GAMES ---")

display(df_genre_distribution_pool)

print(
    "\nNúmero de géneros distintos:",
    df_genre_distribution_pool["genres"].nunique()
)

print(
    "Media de géneros por videojuego:",
    round(df_sampling_pool["genres"].apply(len).mean(), 2)
)

In [ ]:
RANDOM_STATE = 42
SAMPLE_SIZE = 2000

period_quotas = {
    "Antes de 2000": 60,
    "2000-2009": 100,
    "2010-2014": 200,
    "2015-2019": 500,
    "2020-2022": 500,
    "2023-2026": 640
}

assert sum(period_quotas.values()) == SAMPLE_SIZE

# Comprobaciones previas
assert df_sampling_pool["igdb_id"].is_unique
assert df_sampling_pool["appid"].is_unique
assert df_sampling_pool["release_period"].notna().all()

# Selección aleatoria dentro de cada periodo
sample_parts = []

for period, quota in period_quotas.items():

    df_period = df_sampling_pool.loc[
        df_sampling_pool["release_period"] == period
    ]

    assert len(df_period) >= quota, (
        f"No hay suficientes juegos en {period}"
    )

    df_selected = df_period.sample(
        n=quota,
        random_state=RANDOM_STATE
    )

    sample_parts.append(df_selected)

df_sample_initial = pd.concat(
    sample_parts,
    ignore_index=True
)

print("--- MUESTRA INICIAL ---")
print("Juegos:", len(df_sample_initial))
print("IGDB IDs únicos:", df_sample_initial["igdb_id"].nunique())
print("AppIDs únicos:", df_sample_initial["appid"].nunique())

display(
    df_sample_initial["release_period"]
    .value_counts(sort=False)
)

In [ ]:
def calculate_genre_distribution(df):

    df_exploded = (
        df[["igdb_id", "genres"]]
        .explode("genres")
        .dropna(subset=["genres"])
    )

    distribution = (
        df_exploded
        .groupby("genres")["igdb_id"]
        .nunique()
        .rename("n_games")
        .reset_index()
    )

    distribution["percentage"] = (
        distribution["n_games"] / len(df) * 100
    )

    return distribution


# Géneros del universo
df_genres_population = calculate_genre_distribution(
    df_sampling_pool
).rename(
    columns={
        "n_games": "population_games",
        "percentage": "population_pct"
    }
)

# Géneros de la muestra
df_genres_sample = calculate_genre_distribution(
    df_sample_initial
).rename(
    columns={
        "n_games": "sample_games",
        "percentage": "sample_pct"
    }
)

# Comparación
df_genre_comparison = (
    df_genres_population
    .merge(
        df_genres_sample,
        on="genres",
        how="left"
    )
    .fillna({
        "sample_games": 0,
        "sample_pct": 0
    })
)

df_genre_comparison["sample_games"] = (
    df_genre_comparison["sample_games"].astype(int)
)

df_genre_comparison["difference_pp"] = (
    df_genre_comparison["sample_pct"]
    - df_genre_comparison["population_pct"]
)

print("--- REPRESENTACIÓN DE GÉNEROS ---")

display(
    df_genre_comparison.sort_values(
        "population_games",
        ascending=False
    ).round(2)
)

In [ ]:
MIN_GAMES_PER_GENRE = 10

df_genre_comparison["minimum_target"] = (
    df_genre_comparison["population_games"]
    .clip(upper=MIN_GAMES_PER_GENRE)
    .astype(int)
)

df_genre_comparison["deficit"] = (
    df_genre_comparison["minimum_target"]
    - df_genre_comparison["sample_games"]
).clip(lower=0)

df_genres_underrepresented = (
    df_genre_comparison.loc[
        df_genre_comparison["deficit"] > 0
    ]
    .sort_values("deficit", ascending=False)
)

print("--- GÉNEROS POR DEBAJO DEL MÍNIMO ---")

display(
    df_genres_underrepresented[
        [
            "genres",
            "population_games",
            "sample_games",
            "minimum_target",
            "deficit"
        ]
    ]
)

print(
    "\nGéneros que requieren ajustes:",
    len(df_genres_underrepresented)
)

print(
    "Déficit total de apariciones:",
    df_genres_underrepresented["deficit"].sum()
)

In [ ]:
print("--- VALIDACIÓN DE LA MUESTRA INICIAL ---")

print("Tamaño:", len(df_sample_initial))

print(
    "IGDB IDs duplicados:",
    df_sample_initial["igdb_id"].duplicated().sum()
)

print(
    "AppIDs duplicados:",
    df_sample_initial["appid"].duplicated().sum()
)

print(
    "Todos main_game:",
    df_sample_initial["game_type_name"]
    .eq("main_game").all()
)

print(
    "Juegos sin géneros:",
    df_sample_initial["genres"]
    .apply(lambda x: len(x) == 0)
    .sum()
)

print(
    "Juegos con Indie en genres:",
    df_sample_initial["genres"]
    .apply(lambda x: 32 in x)
    .sum()
)

print(
    "Periodos correctos:",
    all(
        (
            df_sample_initial["release_period"]
            .astype(str)
            .value_counts()
            .get(period, 0)
        ) == quota
        for period, quota in period_quotas.items()
    )
)

print(
    "Géneros representados:",
    df_genres_sample["genres"].nunique(),
    "de",
    df_genres_population["genres"].nunique()
)

In [ ]:
# Géneros que queremos reforzar
genre_additions = {
    26: 5,  # Quiz/Trivia
    30: 9,  # Pinball
    36: 5   # MOBA
}

df_sample_final = df_sample_initial.copy()

additional_games = []

for genre_id, n_add in genre_additions.items():

    # Juegos del universo con este género
    df_candidates_genre = df_sampling_pool.loc[
        df_sampling_pool["genres"].apply(
            lambda genres: genre_id in genres
        )
    ].copy()

    # Excluir juegos ya seleccionados
    selected_ids = set(df_sample_final["igdb_id"])

    df_candidates_genre = df_candidates_genre.loc[
        ~df_candidates_genre["igdb_id"].isin(selected_ids)
    ]

    # Selección reproducible
    df_selected_genre = df_candidates_genre.sample(
        n=n_add,
        random_state=RANDOM_STATE + genre_id
    )

    additional_games.append(df_selected_genre)

    # Actualizar muestra para evitar duplicados
    df_sample_final = pd.concat(
        [df_sample_final, df_selected_genre],
        ignore_index=True
    )

print("--- AMPLIACIÓN DE LA MUESTRA ---")

print("Muestra inicial:", len(df_sample_initial))

print(
    "Videojuegos añadidos:",
    sum(len(df) for df in additional_games)
)

print("Muestra final:", len(df_sample_final))

In [ ]:
print("--- VALIDACIÓN FINAL ---")

print("Juegos:", len(df_sample_final))

print(
    "IGDB IDs únicos:",
    df_sample_final["igdb_id"].nunique()
)

print(
    "AppIDs únicos:",
    df_sample_final["appid"].nunique()
)

print(
    "IGDB IDs duplicados:",
    df_sample_final["igdb_id"].duplicated().sum()
)

print(
    "AppIDs duplicados:",
    df_sample_final["appid"].duplicated().sum()
)

print(
    "Todos main_game:",
    df_sample_final["game_type_name"]
    .eq("main_game").all()
)

print("\n--- DISTRIBUCIÓN TEMPORAL FINAL ---")

display(
    df_sample_final["release_period"]
    .value_counts(sort=False)
)

print("\n--- GÉNEROS MINORITARIOS ---")

for genre_id in genre_additions:

    n_games = df_sample_final["genres"].apply(
        lambda genres: genre_id in genres
    ).sum()

    print(f"Género {genre_id}: {n_games} videojuegos")

In [ ]:
from pathlib import Path

OUTPUT_DIR = Path("sampling")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df_sample_final.to_pickle(
    OUTPUT_DIR / "sample_final_2019.pkl"
)

print("Muestra guardada correctamente.")

# Evaluación de la cobertura de nuestra muestra de 2.019 videojuegos en Steam e ITAD

In [ ]:
STEAM_DIR = Path("steam_etl")

manifest_paths = {
    "principal": STEAM_DIR / "manifest.csv",
    "backup": STEAM_DIR / "manifest_backup_20261109_114432.csv",
    "parquet": STEAM_DIR / "parquet_reviews" / "parquet_manifest.csv"
}

for name, path in manifest_paths.items():

    print(f"\n--- MANIFEST: {name.upper()} ---")

    if not path.exists():
        print("No encontrado")
        continue

    df_manifest = pd.read_csv(path)

    print("Filas:", len(df_manifest))
    print("Columnas:", df_manifest.columns.tolist())

    if "appid" in df_manifest.columns:
        print("AppIDs únicos:", df_manifest["appid"].nunique())

    if "status" in df_manifest.columns:
        print("\nEstados:")
        display(df_manifest["status"].value_counts(dropna=False))

    display(df_manifest.head(3))

In [ ]:
REVIEWS_DIR = STEAM_DIR / "reviews"

# Obtener AppIDs a partir de los nombres de archivo
json_appids = set()

for path in REVIEWS_DIR.glob("appid_*.json"):

    try:
        appid = int(path.stem.replace("appid_", ""))
        json_appids.add(appid)

    except ValueError:
        continue

# AppIDs de nuestra muestra
sample_appids = set(
    df_sample_final["appid"].astype(int)
)

# Comparación
available_appids = sample_appids & json_appids
missing_appids = sample_appids - json_appids

print("--- COBERTURA DE STEAM ---")

print("Videojuegos de la muestra:", len(sample_appids))
print("JSON disponibles en Steam:", len(json_appids))

print(
    "Juegos de la muestra con JSON:",
    len(available_appids)
)

print(
    "Juegos de la muestra sin JSON:",
    len(missing_appids)
)

print(
    "Cobertura:",
    round(
        len(available_appids) / len(sample_appids) * 100,
        2
    ),
    "%"
)

In [ ]:
# Cargar manifest principal
df_steam_manifest = pd.read_csv(
    STEAM_DIR / "manifest.csv"
)

df_steam_manifest["appid"] = pd.to_numeric(
    df_steam_manifest["appid"],
    errors="coerce"
).astype("Int64")

# Copia de la muestra
df_sample_steam = df_sample_final.copy()

df_sample_steam["appid"] = (
    pd.to_numeric(
        df_sample_steam["appid"],
        errors="raise"
    ).astype("Int64")
)

# Comprobar existencia de JSON
df_sample_steam["has_json"] = (
    df_sample_steam["appid"].isin(json_appids)
)

# Incorporar información del manifest
df_sample_steam = df_sample_steam.merge(
    df_steam_manifest[
        [
            "appid",
            "status",
            "reviews_downloaded",
            "reviews_reported",
            "elapsed_seconds",
            "error"
        ]
    ].rename(
        columns={
            "status": "steam_status",
            "error": "steam_error"
        }
    ),
    on="appid",
    how="left",
    validate="one_to_one"
)

# Clasificar situación de cada juego
def classify_steam_status(row):

    if row["steam_status"] == "success" and row["has_json"]:
        return "completed"

    if row["steam_status"] == "success" and not row["has_json"]:
        return "missing_json"

    if row["steam_status"] == "failed":
        return "failed"

    if row["has_json"]:
        return "json_without_success"

    return "not_downloaded"


df_sample_steam["steam_coverage_status"] = (
    df_sample_steam.apply(
        classify_steam_status,
        axis=1
    )
)

print("--- ESTADO DE LA MUESTRA EN STEAM ---")

display(
    df_sample_steam["steam_coverage_status"]
    .value_counts()
)

print("\n--- RESEÑAS YA DESCARGADAS ---")

df_completed = df_sample_steam.loc[
    df_sample_steam["steam_coverage_status"] == "completed"
]

print("Juegos completados:", len(df_completed))

print(
    "Reseñas descargadas:",
    int(df_completed["reviews_downloaded"].sum())
)

print(
    "Juegos con cero reseñas:",
    int((df_completed["reviews_downloaded"] == 0).sum())
)

print(
    "Juegos con diferencia respecto al total informado:",
    int(
        (
            df_completed["reviews_reported"].notna()
            & (
                df_completed["reviews_downloaded"]
                != df_completed["reviews_reported"]
            )
        ).sum()
    )
)

In [ ]:
# Seleccionar los juegos pendientes de Steam
df_steam_pending = df_sample_steam.loc[
    df_sample_steam["steam_coverage_status"] == "not_downloaded",
    ["appid"]
].copy()

df_steam_pending["appid"] = (
    df_steam_pending["appid"].astype("int64")
)

df_steam_pending = (
    df_steam_pending
    .drop_duplicates(subset="appid")
    .sort_values("appid")
    .reset_index(drop=True)
)

# Validaciones
assert df_steam_pending["appid"].is_unique

print("--- EXTRACCIONES STEAM PENDIENTES ---")
print("Juegos pendientes:", len(df_steam_pending))

# Guardar CSV
OUTPUT_DIR = Path("sampling")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CSV_PATH = OUTPUT_DIR / "steam_reviews_pending_1689.csv"

df_steam_pending.to_csv(
    CSV_PATH,
    index=False
)

print("CSV guardado:", CSV_PATH.resolve())

##################################################

In [ ]:
ITAD_DIR = Path("itad_etl")

df_itad_coverage = pd.read_csv(
    ITAD_DIR / "itad_coverage_500.csv",
    usecols=[
        "appid",
        "itad_id",
        "current_price",
        "historical_low_price"
    ]
)

df_itad_coverage["appid"] = pd.to_numeric(
    df_itad_coverage["appid"],
    errors="coerce"
).astype("Int64")

print("--- VALIDACIÓN DEL MAPPING ITAD ---")

print("Filas:", len(df_itad_coverage))

print(
    "AppIDs únicos:",
    df_itad_coverage["appid"].nunique()
)

print(
    "AppIDs duplicados:",
    df_itad_coverage["appid"].duplicated().sum()
)

print(
    "ITAD IDs disponibles:",
    df_itad_coverage["itad_id"].notna().sum()
)

print(
    "Precios actuales disponibles:",
    df_itad_coverage["current_price"].notna().sum()
)

print(
    "Mínimos históricos disponibles:",
    df_itad_coverage["historical_low_price"].notna().sum()
)

In [ ]:
print("--- CONSISTENCIA DE APPIDs ---")

df_mapping_consistency = (
    df_itad_coverage
    .groupby("appid")["itad_id"]
    .nunique()
)

print(
    "AppIDs asociados a varios ITAD IDs:",
    (df_mapping_consistency > 1).sum()
)

print(
    "AppIDs sin ITAD ID:",
    df_itad_coverage.loc[
        df_itad_coverage["itad_id"].isna(),
        "appid"
    ].nunique()
)

print("\n--- COBERTURA DE LA MUESTRA ---")

sample_appids = set(
    df_sample_final["appid"].astype(int)
)

mapped_appids = set(
    df_itad_coverage.loc[
        df_itad_coverage["itad_id"].notna(),
        "appid"
    ].dropna().astype(int)
)

current_price_appids = set(
    df_itad_coverage.loc[
        df_itad_coverage["current_price"].notna(),
        "appid"
    ].dropna().astype(int)
)

historical_low_appids = set(
    df_itad_coverage.loc[
        df_itad_coverage["historical_low_price"].notna(),
        "appid"
    ].dropna().astype(int)
)

print("Juegos de la muestra:", len(sample_appids))

print(
    "Con mapping ITAD:",
    len(sample_appids & mapped_appids)
)

print(
    "Con precio actual:",
    len(sample_appids & current_price_appids)
)

print(
    "Con mínimo histórico:",
    len(sample_appids & historical_low_appids)
)

print(
    "Sin mapping ITAD:",
    len(sample_appids - mapped_appids)
)

In [ ]:
df_history_manifest = pd.read_csv(
    ITAD_DIR / "history_manifest.csv"
)

df_history_manifest["appid"] = pd.to_numeric(
    df_history_manifest["appid"],
    errors="coerce"
).astype("Int64")

print("--- HISTORIAL ITAD ---")

print("Registros del manifest:", len(df_history_manifest))

print(
    "AppIDs únicos:",
    df_history_manifest["appid"].nunique()
)

print("\nEstados:")
display(
    df_history_manifest["status"]
    .value_counts(dropna=False)
)

# Juegos con extracción completada
history_success_appids = set(
    df_history_manifest.loc[
        df_history_manifest["status"] == "success",
        "appid"
    ].dropna().astype(int)
)

# Comprobar existencia de JSON solo para la muestra
HISTORY_DIR = ITAD_DIR / "history"

history_available_appids = {
    appid
    for appid in sample_appids & history_success_appids
    if (HISTORY_DIR / f"history_{appid}.json").exists()
}

print("\n--- COBERTURA HISTÓRICA DE LA MUESTRA ---")

print(
    "Con status success:",
    len(sample_appids & history_success_appids)
)

print(
    "Con success y JSON:",
    len(history_available_appids)
)

print(
    "Sin historial validado:",
    len(sample_appids - history_available_appids)
)

print(
    "Con registros históricos descargados:",
    len(
        df_history_manifest.loc[
            df_history_manifest["appid"].isin(
                history_available_appids
            )
            & (df_history_manifest["records_downloaded"] > 0)
        ]["appid"].unique()
    )
)

In [ ]:
# Partimos de la muestra seleccionada
df_extraction_plan = df_sample_final[
    [
        "igdb_id",
        "appid",
        "game_name",
        "release_year",
        "release_period",
        "genres"
    ]
].copy()

df_extraction_plan["appid"] = (
    df_extraction_plan["appid"].astype(int)
)

# Cobertura Steam
df_extraction_plan["steam_completed"] = (
    df_extraction_plan["appid"].isin(
        df_completed["appid"].astype(int)
    )
)

# Cobertura ITAD
df_extraction_plan["itad_mapped"] = (
    df_extraction_plan["appid"].isin(mapped_appids)
)

df_extraction_plan["itad_current_price"] = (
    df_extraction_plan["appid"].isin(current_price_appids)
)

df_extraction_plan["itad_historical_low"] = (
    df_extraction_plan["appid"].isin(historical_low_appids)
)

df_extraction_plan["itad_history_completed"] = (
    df_extraction_plan["appid"].isin(
        history_available_appids
    )
)

df_extraction_plan["itad_history_has_records"] = (
    df_extraction_plan["appid"].isin(
        df_history_manifest.loc[
            (df_history_manifest["status"] == "success")
            & (df_history_manifest["records_downloaded"] > 0),
            "appid"
        ].dropna().astype(int)
    )
)

# Juegos con ambas extracciones completadas
df_extraction_plan["both_completed"] = (
    df_extraction_plan["steam_completed"]
    & df_extraction_plan["itad_history_completed"]
)

print("--- COBERTURA CONJUNTA ---")

print(
    "Steam + historial ITAD completados:",
    df_extraction_plan["both_completed"].sum()
)

print(
    "Steam + historial ITAD con registros:",
    (
        df_extraction_plan["steam_completed"]
        & df_extraction_plan["itad_history_has_records"]
    ).sum()
)

print(
    "Sin ninguna de las dos extracciones:",
    (
        ~df_extraction_plan["steam_completed"]
        & ~df_extraction_plan["itad_history_completed"]
    ).sum()
)

In [ ]:
# 1. Identificar los juegos pendientes
pending_appids = (
    set(df_sample_final["appid"].astype(int))
    - history_available_appids
)

# 2. Crear DataFrame con TODOS los pendientes
df_itad_pending = pd.DataFrame({
    "appid": sorted(pending_appids)
})

# 3. Incorporar los ITAD IDs disponibles
df_itad_pending = df_itad_pending.merge(
    df_itad_coverage[["appid", "itad_id"]],
    on="appid",
    how="left",
    validate="one_to_one"
)

# 4. Validaciones
assert df_itad_pending["appid"].is_unique

print("--- EXTRACCIONES ITAD PENDIENTES ---")

print("Juegos pendientes:", len(df_itad_pending))

print(
    "Con ITAD ID:",
    df_itad_pending["itad_id"].notna().sum()
)

print(
    "Sin ITAD ID:",
    df_itad_pending["itad_id"].isna().sum()
)

# 5. Guardar CSV
OUTPUT_DIR = Path("sampling")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CSV_PATH = OUTPUT_DIR / "itad_history_pending_1679.csv"

df_itad_pending.to_csv(
    CSV_PATH,
    index=False
)

print("\nCSV guardado:", CSV_PATH.resolve())

############################################################

# Limpieza de IGDB

1. Comprobaremos duplicados por id

2. Excluiremos fechas futuras del análisis histórico

3. Identificaremos game_type y separaremos juegos principales de DLCs, expansiones, etc

4. Excluiremos demos, betas, playtests, etc... claramente identificables

5. Normalizaremos géneros, plataformas, y perspectiva de jugador

6. Gestionaremos valores nulos

In [ ]:
df_igdb = pd.read_json("igdb_games_raw.json") # df_games

# Crear copia de trabajo sin modificar el dataset original
df_igdb_clean = df_igdb.copy()

print("--- DIMENSIONES ---")
print("Filas:", len(df_igdb_clean))
print("Columnas:", len(df_igdb_clean.columns))

print("\n--- COLUMNAS Y TIPOS DE DATOS ---")
display(df_igdb_clean.dtypes.to_frame("dtype"))

print("\n--- PRIMEROS REGISTROS ---")
display(df_igdb_clean.head(3))

In [ ]:
print("--- DUPLICADOS IGDB ---")

print("Filas totales:", len(df_igdb_clean))

print(
    "IDs únicos:",
    df_igdb_clean["id"].nunique()
)

print(
    "IDs duplicados:",
    df_igdb_clean["id"].duplicated().sum()
)

print(
    "IDs nulos:",
    df_igdb_clean["id"].isna().sum()
)

# Mostrar juegos con IDs duplicados
df_duplicates = df_igdb_clean.loc[
    df_igdb_clean["id"].duplicated(keep=False)
].sort_values("id")

display(
    df_duplicates[
        ["id", "name", "first_release_date", "game_type"]
    ].head(20)
)

In [ ]:
df_nulls = pd.DataFrame({
    "null_count": df_igdb_clean.isna().sum(),
    "null_pct": (
        df_igdb_clean.isna().mean() * 100
    ).round(2)
})

df_nulls = df_nulls.sort_values(
    "null_pct",
    ascending=False
)

display(df_nulls)

In [ ]:
# columnas a eliminar del dataset debido a la cantidad de nulos o falta de relevancia para el análisis
columns_to_drop = [
    "rating",
    "rating_count",
    "aggregated_rating",
    "aggregated_rating_count",
    "total_rating",
    "total_rating_count",
    "hypes",
    "keywords",
    "themes",
    "summary",
    "slug"#,
    #"dlcs",
    #"expansions"
]

df_igdb_clean = df_igdb_clean.drop(
    columns=columns_to_drop,
    errors="ignore"
)

print("--- COLUMNAS RESTANTES ---")
print(df_igdb_clean.columns.tolist())

print("\nDimensiones:", df_igdb_clean.shape)

In [ ]:
# Convertimos las fechas como timestamps Unix en segundos
df_igdb_clean["first_release_date"] = pd.to_datetime(
    df_igdb_clean["first_release_date"],
    unit="s",
    errors="coerce"
)

In [ ]:
df_igdb_clean["release_year"] = (
    df_igdb_clean["first_release_date"].dt.year
)

print("--- FECHAS DE LANZAMIENTO ---")

print(
    "Fecha mínima:",
    df_igdb_clean["first_release_date"].min()
)

print(
    "Fecha máxima:",
    df_igdb_clean["first_release_date"].max()
)

print(
    "Juegos sin fecha:",
    df_igdb_clean["release_year"].isna().sum()
)

print(
    "Juegos con fecha futura (2027+):",
    (df_igdb_clean["release_year"] >= 2027).sum()
)

display(
    df_igdb_clean["release_year"]
    .value_counts(dropna=False)
    .sort_index()
    .tail(15)
)

In [ ]:
print("--- TIPOS DE JUEGO ---")

if "game_type" in df_igdb_clean.columns:
    display(
        df_igdb_clean["game_type"]
        .value_counts(dropna=False)
    )

In [ ]:
game_type_names = {
    0: "main_game",
    1: "dlc_addon",
    2: "expansion",
    3: "bundle",
    4: "standalone_expansion",
    5: "mod",
    6: "episode",
    7: "season",
    8: "remake",
    9: "remaster",
    10: "expanded_game",
    11: "port",
    12: "fork",
    13: "pack",
    14: "update"
}

df_igdb_clean["game_type_name"] = (
    df_igdb_clean["game_type"].map(game_type_names)
)

print("--- DISTRIBUCIÓN DE TIPOS DE JUEGO ---")

display(
    df_igdb_clean["game_type_name"]
    .value_counts(dropna=False)
)

In [ ]:
df_genres = pd.read_csv("igdb_genres_names.csv")
df_platforms = pd.read_csv("igdb_platforms_names.csv")
df_game_modes = pd.read_csv("igdb_game_modes_names.csv")
df_player_perspectives = pd.read_csv(
    "igdb_player_perspectives_names.csv"
)

# Diccionarios ID -> nombre
genre_lookup = dict(zip(df_genres["id"], df_genres["name"]))

platform_lookup = dict(
    zip(df_platforms["id"], df_platforms["name"])
)

game_mode_lookup = dict(
    zip(df_game_modes["id"], df_game_modes["name"])
)

perspective_lookup = dict(
    zip(
        df_player_perspectives["id"],
        df_player_perspectives["name"]
    )
)

In [ ]:
def map_ids_to_names(values, lookup):
    """
    Convierte listas de IDs de IGDB en listas de nombres.
    Conserva la estructura multietiqueta.
    """

    if not isinstance(values, (list, tuple, np.ndarray)):
        return []

    names = []

    for value in values:

        if isinstance(value, dict):
            name = value.get("name")

            if name is None and "id" in value:
                name = lookup.get(value["id"])

        elif isinstance(value, (int, np.integer)):
            name = lookup.get(int(value))

        elif isinstance(value, str) and value.isdigit():
            name = lookup.get(int(value))

        else:
            name = None

        if name is not None:
            names.append(str(name))

    return list(dict.fromkeys(names))

In [ ]:
# Aplicamos la normalización a las cuatro variables
df_igdb_clean["genres_names"] = (
    df_igdb_clean["genres"].apply(
        lambda x: map_ids_to_names(x, genre_lookup)
    )
)

df_igdb_clean["platforms_names"] = (
    df_igdb_clean["platforms"].apply(
        lambda x: map_ids_to_names(x, platform_lookup)
    )
)

df_igdb_clean["game_modes_names"] = (
    df_igdb_clean["game_modes"].apply(
        lambda x: map_ids_to_names(x, game_mode_lookup)
    )
)

df_igdb_clean["player_perspectives_names"] = (
    df_igdb_clean["player_perspectives"].apply(
        lambda x: map_ids_to_names(x, perspective_lookup)
    )
)

In [ ]:
# Clasificación Indie según IGDB
df_igdb_clean["is_indie"] = (
    df_igdb_clean["genres_names"].apply(
        lambda genres: "Indie" in genres
    )
)

# Géneros sin la etiqueta Indie
df_igdb_clean["genres_clean"] = (
    df_igdb_clean["genres_names"].apply(
        lambda genres: [
            genre
            for genre in genres
            if genre != "Indie"
        ]
    )
)

print("--- CLASIFICACIÓN INDIE ---")

display(
    df_igdb_clean["is_indie"]
    .value_counts(dropna=False)
)

print("\nEjemplos:")

display(
    df_igdb_clean[
        [
            "name",
            "genres_names",
            "genres_clean",
            "is_indie"
        ]
    ].head(10)
)

In [ ]:
columns = [
    "genres_clean",
    "platforms_names",
    "game_modes_names",
    "player_perspectives_names"
]

print("--- COBERTURA DE VARIABLES MULTIETIQUETA ---")

for col in columns:

    lengths = df_igdb_clean[col].apply(len)

    print(f"\n{col}")

    print("Con información:", (lengths > 0).sum())
    print("Sin información:", (lengths == 0).sum())
    print("Con múltiples etiquetas:", (lengths > 1).sum())
    print("Máximo de etiquetas:", lengths.max())

In [ ]:
def count_unmapped_ids(values, lookup):

    if not isinstance(values, (list, tuple, np.ndarray)):
        return 0

    count = 0

    for value in values:

        if isinstance(value, dict):
            if value.get("name") is not None:
                continue
            value = value.get("id")

        try:
            if int(value) not in lookup:
                count += 1
        except (ValueError, TypeError):
            count += 1

    return count


print("--- IDs SIN CORRESPONDENCIA ---")

mapping_config = {
    "genres": genre_lookup,
    "platforms": platform_lookup,
    "game_modes": game_mode_lookup,
    "player_perspectives": perspective_lookup
}

for col, lookup in mapping_config.items():

    unmapped = df_igdb_clean[col].apply(
        lambda x: count_unmapped_ids(x, lookup)
    ).sum()

    print(f"{col}: {unmapped}")

In [ ]:
df_genre_counts = (
    df_igdb_clean[["id", "genres_clean"]]
    .explode("genres_clean")
    .dropna(subset=["genres_clean"])
    .groupby("genres_clean")["id"]
    .nunique()
    .sort_values(ascending=False)
)

display(df_genre_counts.head(15))

In [ ]:
print("--- DUPLICADOS POR ID ---")

print("Registros totales:", len(df_igdb_clean))
print("IDs únicos:", df_igdb_clean["id"].nunique())
print("IDs nulos:", df_igdb_clean["id"].isna().sum())
print("IDs duplicados:", df_igdb_clean["id"].duplicated().sum())

# Juegos con IDs repetidos
df_duplicate_ids = df_igdb_clean.loc[
    df_igdb_clean["id"].duplicated(keep=False)
].sort_values("id")

display(
    df_duplicate_ids[
        ["id", "name", "game_type_name", "first_release_date"]
    ].head(20)
)

In [ ]:
exclusion_patterns = {
    "playtest": r"\bplay[\s-]?test\b",
    "demo": r"\bdemo\b",
    "beta": r"\bbeta\b",
    "alpha": r"\balpha\b",
    "tech_demo": r"\btech[\s-]?demo\b",
    "prologue": r"\bprologue\b"
}

In [ ]:
def detect_exclusion_patterns(name):

    if pd.isna(name):
        return []

    name = str(name)

    matches = []

    for category, pattern in exclusion_patterns.items():

        if re.search(pattern, name, flags=re.IGNORECASE):
            matches.append(category)

    return matches


df_igdb_clean["exclusion_reasons"] = (
    df_igdb_clean["name"].apply(detect_exclusion_patterns)
)

df_igdb_clean["is_potential_exclusion"] = (
    df_igdb_clean["exclusion_reasons"].apply(bool)
)

In [ ]:
print("--- POSIBLES EXCLUSIONES ---")

print(
    "Total de juegos identificados:",
    df_igdb_clean["is_potential_exclusion"].sum()
)

print("\nPor categoría:")

display(
    df_igdb_clean["exclusion_reasons"]
    .explode()
    .dropna()
    .value_counts()
)

print("\nPor tipo de juego:")

display(
    pd.crosstab(
        df_igdb_clean["game_type_name"],
        df_igdb_clean["is_potential_exclusion"]
    )
)

In [ ]:
df_exclusion_candidates = df_igdb_clean.loc[
    (df_igdb_clean["is_potential_exclusion"])
    & (df_igdb_clean["game_type_name"] == "main_game")
].copy()

print(
    "Main games potencialmente excluibles:",
    len(df_exclusion_candidates)
)

display(
    df_exclusion_candidates[
        [
            "id",
            "name",
            "game_type_name",
            "release_year",
            "exclusion_reasons"
        ]
    ].sample(
        n=min(30, len(df_exclusion_candidates)),
        random_state=42
    )
)

In [ ]:
sample_ids = set(
    df_sample_final["igdb_id"].dropna().astype(int)
)

df_sample_exclusions = df_igdb_clean.loc[
    df_igdb_clean["id"].isin(sample_ids)
    & df_igdb_clean["is_potential_exclusion"]
].copy()

print(
    "Juegos de la muestra identificados:",
    len(df_sample_exclusions)
)

display(
    df_sample_exclusions[
        [
            "id",
            "name",
            "game_type_name",
            "exclusion_reasons"
        ]
    ]
)

In [ ]:
print("Steam mapping:")
print(df_steam_map_clean.columns.tolist())

print("\nSteam manifest:")
print(pd.read_csv("steam_etl/manifest.csv", nrows=2).columns.tolist())

print("\nITAD manifest:")
print(pd.read_csv("itad_etl/history_manifest.csv", nrows=2).columns.tolist())

In [ ]:
# Juegos de la muestra estratificada
sample_ids = set(
    df_sample_final["igdb_id"].dropna().astype(int)
)

# Steam: extracciones completadas con JSON
steam_manifest = pd.read_csv("steam_etl/manifest.csv")

steam_appids = {
    int(appid)
    for appid in steam_manifest.loc[
        steam_manifest["status"] == "success", "appid"
    ].dropna()
    if (Path("steam_etl/reviews") / f"appid_{int(appid)}.json").exists()
}

# ITAD: extracciones completadas con JSON
itad_manifest = pd.read_csv("itad_etl/history_manifest.csv")

itad_appids = {
    int(appid)
    for appid in itad_manifest.loc[
        itad_manifest["status"] == "success", "appid"
    ].dropna()
    if (Path("itad_etl/history") / f"history_{int(appid)}.json").exists()
}

# Unir los AppIDs protegidos
protected_appids = steam_appids | itad_appids

# Convertir AppIDs a IDs de IGDB
protected_igdb_ids = set(
    df_steam_map_clean.loc[
        df_steam_map_clean["uid"].isin(protected_appids),
        "game"
    ].dropna().astype(int)
)

# Añadir la muestra
protected_igdb_ids |= sample_ids

print("Juegos protegidos (IGDB IDs):", len(protected_igdb_ids))
print("Juegos de la muestra:", len(sample_ids))

In [ ]:
# Categorías que consideramos excluibles
exclude_categories = {
    "demo",
    "tech_demo",
    "playtest",
    "prologue"
}

# Identificar juegos con alguna de esas etiquetas
df_igdb_clean["exclude_candidate"] = (
    df_igdb_clean["exclusion_reasons"].apply(
        lambda reasons: bool(
            set(reasons) & exclude_categories
        )
    )
)

# Identificar juegos protegidos
df_igdb_clean["is_protected"] = (
    df_igdb_clean["id"].isin(protected_igdb_ids)
)

# Exclusión definitiva
df_igdb_clean["exclude_final"] = (
    df_igdb_clean["exclude_candidate"]
    & ~df_igdb_clean["is_protected"]
)

In [ ]:
print("--- RESULTADOS DE LIMPIEZA ---")

print("Registros originales:", len(df_igdb_clean))

print(
    "Candidatos a exclusión:",
    df_igdb_clean["exclude_candidate"].sum()
)

print(
    "Candidatos protegidos:",
    (
        df_igdb_clean["exclude_candidate"]
        & df_igdb_clean["is_protected"]
    ).sum()
)

print(
    "Registros a eliminar:",
    df_igdb_clean["exclude_final"].sum()
)

print("\n--- EJEMPLOS DE EXCLUSIONES ---")

display(
    df_igdb_clean.loc[
        df_igdb_clean["exclude_final"],
        ["id", "name", "game_type_name", "exclusion_reasons"]
    ].head(30)
)

In [ ]:
assert not df_igdb_clean.loc[
    df_igdb_clean["id"].isin(sample_ids),
    "exclude_final"
].any()

print("Validación correcta: ningún juego de la muestra será eliminado.")

In [ ]:
df_igdb_clean = (
    df_igdb_clean.loc[
        ~df_igdb_clean["exclude_final"]
    ]
    .drop(
        columns=[
            "exclusion_reasons",
            "is_potential_exclusion",
            "exclude_candidate",
            "is_protected",
            "exclude_final"
        ],
        errors="ignore"
    )
    .reset_index(drop=True)
)

print("Dataset limpio:", df_igdb_clean.shape)

assert df_igdb_clean["id"].is_unique
assert sample_ids.issubset(set(df_igdb_clean["id"]))

print("Limpieza completada.")

In [ ]:
print("--- CONSTRUCCIÓN DE DF_GAMES ---")

print("IGDB limpio:", len(df_igdb_clean))

print(
    "Main games:",
    (df_igdb_clean["game_type"] == 0).sum()
)

df_main = df_igdb_clean.loc[
    df_igdb_clean["game_type"] == 0
]

print(
    "Main games sin fecha:",
    df_main["release_year"].isna().sum()
)

print(
    "Main games futuros (2027+):",
    (df_main["release_year"] >= 2027).sum()
)

print(
    "Main games válidos hasta 2026:",
    (
        df_main["release_year"].notna()
        & (df_main["release_year"] <= 2026)
    ).sum()
)

In [ ]:
df_games_clean = df_igdb_clean.loc[
    (df_igdb_clean["game_type"] == 0)
    & (df_igdb_clean["release_year"].notna())
    & (df_igdb_clean["release_year"] <= 2026)
].copy()

df_games_clean = df_games_clean.reset_index(drop=True)

print("Dimensiones de df_games_clean:", df_games_clean.shape)

print(
    "Rango de años:",
    df_games_clean["release_year"].min(),
    "-",
    df_games_clean["release_year"].max()
)

print(
    "IDs únicos:",
    df_games_clean["id"].nunique()
)

In [ ]:
sample_ids = set(
    df_sample_final["igdb_id"].astype(int)
)

games_ids = set(df_games_clean["id"].astype(int))

missing_sample_ids = sample_ids - games_ids

print("--- VALIDACIÓN DE LA MUESTRA ---")

print("Juegos de la muestra:", len(sample_ids))

print(
    "Juegos presentes en df_games_clean:",
    len(sample_ids & games_ids)
)

print(
    "Juegos de la muestra ausentes:",
    len(missing_sample_ids)
)

if missing_sample_ids:
    display(
        df_igdb_clean.loc[
            df_igdb_clean["id"].isin(missing_sample_ids),
            ["id", "name", "game_type_name", "release_year"]
        ]
    )

In [ ]:
df_games_clean.isnull().sum()

In [ ]:
df_games_clean["release_year"] = (
    df_games_clean["release_year"].astype(int)
)

In [ ]:
today = pd.Timestamp.today().normalize()

df_games_clean = df_games_clean.loc[
    df_games_clean["first_release_date"] <= today
].copy().reset_index(drop=True)

print("Juegos publicados:", len(df_games_clean))
print("Última fecha:", df_games_clean["first_release_date"].max())

In [ ]:
# Carpeta para datasets preparados para la EDA
EDA_DIR = Path("eda_data")
EDA_DIR.mkdir(parents=True, exist_ok=True)

datasets = {
    "igdb_clean": df_igdb_clean,
    "games_clean": df_games_clean,
    "sample_final": df_sample_final
}

for name, df in datasets.items():

    output_path = EDA_DIR / f"{name}.parquet"

    df.to_parquet(
        output_path,
        index=False,
        engine="pyarrow"
    )

    print(
        f"{name}: {df.shape[0]:,} filas, "
        f"{df.shape[1]} columnas"
    )

print("\nDatasets guardados en:", EDA_DIR.resolve())

#######################################################

In [ ]:
STEAM_PARQUET_MANIFEST = Path(
    "steam_etl/parquet_reviews/parquet_manifest.csv"
)

ITAD_PARQUET_MANIFEST = Path(
    "itad_etl/parquet_history/parquet_manifest.csv"
)

df_steam_parquet = pd.read_csv(STEAM_PARQUET_MANIFEST)
df_itad_parquet = pd.read_csv(ITAD_PARQUET_MANIFEST)

print("--- STEAM ---")
print(df_steam_parquet["status"].value_counts())

print("\n--- ITAD ---")
print(df_itad_parquet["status"].value_counts())

In [ ]:
# Steam: conversiones completadas
steam_appids = set(
    pd.to_numeric(
        df_steam_parquet.loc[
            df_steam_parquet["status"].eq("success")
            & df_steam_parquet["parquet_parts"].gt(0),
            "appid"
        ],
        errors="coerce"
    ).dropna().astype(int)
)

# ITAD: conversiones completadas con historial
itad_appids = set(
    pd.to_numeric(
        df_itad_parquet.loc[
            df_itad_parquet["status"].eq("success")
            & df_itad_parquet["parquet_parts"].gt(0),
            "appid"
        ],
        errors="coerce"
    ).dropna().astype(int)
)

print("Steam con Parquet:", len(steam_appids))
print("ITAD con Parquet:", len(itad_appids))
print("AppIDs presentes en ambas:", len(steam_appids & itad_appids))

In [ ]:
df_mapping = (
    df_steam_map_clean[
        ["game", "uid"]
    ]
    .copy()
)

df_mapping["game"] = pd.to_numeric(
    df_mapping["game"], errors="coerce"
)

df_mapping["uid"] = pd.to_numeric(
    df_mapping["uid"], errors="coerce"
)

df_mapping = (
    df_mapping
    .dropna(subset=["game", "uid"])
    .astype({"game": "int64", "uid": "int64"})
    .drop_duplicates()
)

print("Correspondencias IGDB-Steam:", len(df_mapping))
print("Juegos IGDB únicos:", df_mapping["game"].nunique())
print("AppIDs únicos:", df_mapping["uid"].nunique())

In [ ]:
df_coverage = (
    df_games_clean[
        ["id", "name", "release_year", "is_indie"]
    ]
    .merge(
        df_mapping,
        left_on="id",
        right_on="game",
        how="left"
    )
)

df_coverage["has_steam"] = (
    df_coverage["uid"].isin(steam_appids)
)

df_coverage["has_itad"] = (
    df_coverage["uid"].isin(itad_appids)
)

print("Filas después del JOIN:", len(df_coverage))
print("Juegos IGDB únicos:", df_coverage["id"].nunique())

In [ ]:
df_coverage_games = (
    df_coverage
    .groupby("id", as_index=False)
    .agg(
        name=("name", "first"),
        release_year=("release_year", "first"),
        is_indie=("is_indie", "first"),
        has_steam=("has_steam", "any"),
        has_itad=("has_itad", "any")
    )
)

df_coverage_games["has_both"] = (
    df_coverage_games["has_steam"]
    & df_coverage_games["has_itad"]
)

total_games = len(df_coverage_games)

print("--- COBERTURA ENTRE LAS 3 FUENTES ---")

print("Juegos principales IGDB:", total_games)

print(
    "Con reseñas Steam:",
    df_coverage_games["has_steam"].sum()
)

print(
    "Con historial ITAD:",
    df_coverage_games["has_itad"].sum()
)

print(
    "Con Steam + ITAD:",
    df_coverage_games["has_both"].sum()
)

print(
    "Sin datos de ninguna de las dos:",
    (
        ~df_coverage_games["has_steam"]
        & ~df_coverage_games["has_itad"]
    ).sum()
)

In [ ]:
df_coverage["same_appid_both"] = (
    df_coverage["has_steam"]
    & df_coverage["has_itad"]
)

games_same_appid = (
    df_coverage.loc[
        df_coverage["same_appid_both"],
        "id"
    ].nunique()
)

print(
    "Juegos con Steam e ITAD en el mismo AppID:",
    games_same_appid
)

print(
    "Juegos presentes en ambas fuentes, "
    "pero sin AppID común:",
    int(df_coverage_games["has_both"].sum())
    - games_same_appid
)

In [ ]:
coverage_summary = pd.DataFrame({
    "Source": [
        "IGDB",
        "IGDB + Steam",
        "IGDB + ITAD",
        "IGDB + Steam + ITAD (same AppID)"
    ],
    "Games": [
        total_games,
        int(df_coverage_games["has_steam"].sum()),
        int(df_coverage_games["has_itad"].sum()),
        games_same_appid
    ]
})

coverage_summary["Coverage (%)"] = (
    coverage_summary["Games"]
    / total_games
    * 100
).round(2)

display(coverage_summary)